<div align="center">
  <h2>ResNet50 pretrained ImageNet</h2>
  <p><strong>Klasifikasi pneumonia dari chest X-ray</strong></p>
  <p>Laporan eksperimen tersimpan | PyTorch CUDA | 28 Oktober 2026</p>
</div>

- **Muhammad Arsyad Setiawan** 
- **Amanda Olivia Safitri** 


---

## Abstrak
<a id="abstrak"></a>

Notebook ini menyediakan **ResNet50 pretrained ImageNet-1K V2 tanpa attention tambahan**, dengan fine-tuning seluruh layer, sebagai pembanding bagi penelitian CNN ringan yang mengacu pada [paper PneuNet](https://doi.org/10.3389/fmed.2025.1713587). Tujuan evaluasinya adalah menilai performa klasifikasi bersama biaya inferensi, bukan hanya mencari accuracy tertinggi. Dataset chest X-ray memuat **5.856 citra: 4.646 train, 586 validation, dan 624 test**. Dua kondisi diuji: letterbox 224 x 224 dan normalisasi sebagai baseline, serta CLAHE dengan distribusi train 50% tanpa augmentasi geometrik/blur, 25% rotasi, dan 25% Gaussian blur. Seluruh gambar pada cabang kedua mendapat CLAHE; validation dan test tidak diberi augmentasi acak.

Analisis menggunakan **2 eksperimen full yang sudah tersimpan**, termasuk metrik per kelas, kurva train-validation loss, Grad-CAM, dan benchmark inferensi batch 1 pada RTX 3060 Laptop GPU (FP32). Model memiliki **23,510 juta parameter**. Baseline mencapai accuracy **80,13%** (macro-F1 **0,7555**) dengan latency forward GPU **11,42 ms/gambar**. Kondisi CLAHE+aug mencapai accuracy **83,65%** (macro-F1 **0,8093**), meningkat **3,53 poin persentase**, dengan latency GPU **11,54 ms/gambar** dan Tensor-E2E **11,82 ms/gambar**.

Hasil menyediakan acuan performa dan biaya inferensi model pretrained untuk dibandingkan dengan varian PneuNet pada notebook pendamping. Karena notebook ini menguji satu arsitektur, hasilnya sendiri belum membuktikan bahwa CNN ringan menyamai model pretrained. Latency antar-run hanya bersifat deskriptif sampai diuji dalam benchmark bersama; selisih latency kedua kondisi juga tidak mengukur biaya CLAHE karena preprocessing di luar timer. Tidak ada training atau benchmark ulang dalam penyusunan laporan ini.


> **Versi repository:** output citra, checkpoint, dan dataset tidak disertakan. Semua path mengikuti root repository. Default `RUN_TRAINING=False`; pilih alur preprocessing + smoke/full atau isi `SAVED_RUN_DIR` untuk laporan historis sesuai [README](README.md). Angka pada abstrak/kesimpulan berasal dari eksperimen lokal terdahulu, bukan eksekusi notebook publik ini.

# Daftar isi

1. [Abstrak](#abstrak)
2. [Environment dan konfigurasi](#configuration)
3. [Dataset](#dataset)
4. [Exploratory Data Analysis](#eda)
5. [Preprocessing dan augmentasi](#preprocessing)
6. [Modeling dan protokol eksperimen](#modeling)
7. [Hasil evaluasi dan kurva loss](#hasil-evaluasi)
   - [Confusion matrix dan classification report](#classification-reports)
8. [Analisis kesalahan dan Grad-CAM](#gradcam)
9. [Efisiensi inferensi tersimpan](#benchmark)
10. [Kesimpulan](#kesimpulan)
11. [Referensi](#references)

**Alur laporan:** inventaris data, EDA, keputusan preprocessing, spesifikasi model, hasil tersimpan,
visualisasi error, kemudian kesimpulan berdasarkan angka yang benar-benar tercatat.

# Environment dan konfigurasi
<a id="configuration"></a>

Notebook dibuka dalam **mode laporan**. `RUN_TRAINING=False` memblokir sel eksekusi training,
dan `REBUILD_PREPROCESSING=False` memakai array cache yang sudah tersedia. Definisi model dan training loop
tetap disertakan untuk dokumentasi, tetapi tidak dipanggil saat laporan dijalankan.

Sel kode pertama memuat seluruh pengaturan. `SAVED_RUN_DIR` menentukan run historis yang sesuai arsitektur,
sedangkan `REPORT_OUTPUT_DIR` menampung grafik turunan. Kernel yang dipakai adalah **ai_env (Python 3.10.20)**.

In [ ]:
# Semua konfigurasi eksperimen diubah di sel ini, kemudian jalankan ulang dari atas.
from pathlib import Path
import numpy as np

# Lokasi dan identitas data
WORKSPACE = Path.cwd().resolve()  # Jalankan Jupyter dari root repository.
ROOT = WORKSPACE / "Datasets" / "chest_xray"
CACHE_ROOT = WORKSPACE / "tmp" / "ablation_cache"
PREPROCESSED_ARCHIVE = WORKSPACE / "pneunet_prep.npz"
MODELING_WORKSPACE = WORKSPACE
MODELING_RESULTS_ROOT = WORKSPACE / "outputs" / "modeling"

# Laporan dari checkpoint tersimpan. Training tidak dijalankan pada mode awal.
RUN_TRAINING = False
REBUILD_PREPROCESSING = False
SAVED_RUN_DIR = WORKSPACE / "outputs" / "modeling" / "saved_full_v3"
REPORT_OUTPUT_DIR = WORKSPACE / "outputs" / "notebook_reports" / 'resnet50_pretrained_v3'
REPORT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Confusion matrix dan classification report dari prediksi tersimpan saja.
CLASSIFICATION_REPORT_SPLITS = ("train", "val", "test")
CLASSIFICATION_REPORT_EXPORT = True

SEED = 42
IMG_SIZE = 224
CLASS_NAMES = ["NORMAL", "PNEUMONIA"]
LABEL2ID = {name: i for i, name in enumerate(CLASS_NAMES)}
SPLIT_NAMES = ("train", "val", "test")
EXPECTED_SPLIT_COUNTS = {"train": 4646, "val": 586, "test": 624}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tif", ".tiff", ".webp"}

# EDA dan preprocessing
EDA_SAMPLES_PER_CLASS = 5
EDA_INTENSITY_SAMPLES = 300
PREPROCESS_CHUNK_SIZE = 64
CLAHE_CLIP_LIMIT = 2.5
CLAHE_TILE_GRID_SIZE = (8, 8)
AUGMENTATION_NAMES = ("original", "rotation", "gaussian")
AUGMENTATION_LABELS = ("CLAHE saja", "CLAHE + rotasi", "CLAHE + Gaussian blur")
AUGMENTATION_PROBS = np.array([0.50, 0.25, 0.25])
ROTATION_MAX_DEGREES = 15.0
GAUSSIAN_SIGMA_RANGE = (0.5, 1.5)

# Modeling: default hanya smoke test, full training dijalankan sendiri oleh pengguna.
MODELING_RUN_MODE = "smoke"
CPU_THREADS = 2
DEVICE_REQUEST = "cuda:0"  # GPU wajib tersedia.
EXPECTED_PYTHON = None  # Opsional: Path interpreter tertentu; tidak dikunci ke komputer penulis.
DATALOADER_WORKERS = 0  # Aman untuk Dataset dalam notebook Windows.
PIN_MEMORY = True
DETERMINISTIC = True
CUBLAS_WORKSPACE = ":4096:8"
ARCH_VERSION = "resnet50_imagenet1k_v2_binary_finetune_v1"
MODEL_NAMES = ("ResNet50_Pretrained",)
DATA_CONDITIONS = ("baseline", "clahe_aug")
SELECTED_MODELS = list(MODEL_NAMES)
SELECTED_CONDITIONS = list(DATA_CONDITIONS)
TRAIN_CONFIG = {
    "seed": SEED, "image_size": IMG_SIZE, "batch_size": 32, "epochs": 50,
    "learning_rate": 0.001, "beta_1": 0.9, "beta_2": 0.999, "epsilon": 1e-7,
    "dropout": 0.3, "l2": 0.001, "early_stopping_patience": 7,
    "micro_batch_size": 4,  # Effective batch 32; statistik BN memakai micro-batch.
    "threshold": 0.5, "initial_weights": "IMAGENET1K_V2",
}
SMOKE_CONFIG = {"train_samples": 2, "validation_samples": 2, "batch_size": 2,
                "epochs": 1, "steps_per_epoch": 1, "validation_steps": 1}

# Error analysis: satu checkpoint hasil run di atas, tanpa training tambahan.
ERROR_ANALYSIS_MODEL = "ResNet50_Pretrained"
ERROR_ANALYSIS_CONDITION = "baseline"
ERROR_ANALYSIS_MAX_CASES = 6
GRADCAM_ALPHA = 0.70
GRADCAM_TARGET_LAYERS = {"ResNet50_Pretrained": "backbone.layer4"}
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


assert MODELING_RUN_MODE in ("smoke", "full")
assert IMG_SIZE == 224, "Reproduksi ini mengikuti input 224x224 pada paper."
assert np.isclose(AUGMENTATION_PROBS.sum(), 1) and (AUGMENTATION_PROBS >= 0).all()
assert set(SELECTED_MODELS).issubset(MODEL_NAMES) and SELECTED_MODELS
assert set(SELECTED_CONDITIONS).issubset(DATA_CONDITIONS) and SELECTED_CONDITIONS
assert ERROR_ANALYSIS_MAX_CASES > 0 and 0 <= GRADCAM_ALPHA <= 1
print(f"Mode: {MODELING_RUN_MODE.upper()} | seed={SEED} | input={IMG_SIZE}x{IMG_SIZE}")
print("Dataset:", ROOT)
print("Mode notebook: laporan tersimpan | RUN_TRAINING=", RUN_TRAINING)


## Import pustaka dan inisialisasi

Pustaka dimuat sekali dari kernel ai_env. Seed Python, NumPy, dan PyTorch mengikuti konfigurasi. Sel berhenti jika interpreter berbeda atau CUDA tidak tersedia.

In [ ]:
import os
import sys
import re
import gc
import json
import math
import random
import shutil
import hashlib
import tempfile
import zipfile
import time
from datetime import datetime

# Conda DLL search path before SciPy/OpenCV/Torch on Windows.
_conda_bin = Path(sys.prefix) / "Library" / "bin"
if os.name == "nt" and _conda_bin.is_dir():
    os.environ["PATH"] = str(_conda_bin) + os.pathsep + os.environ.get("PATH", "")
    _conda_dll_handle = os.add_dll_directory(str(_conda_bin))
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", CUBLAS_WORKSPACE)
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import models
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import Image as NotebookImage, display
from sklearn.metrics import (
    accuracy_score, average_precision_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
print("Python:", sys.executable)
print("PyTorch:", torch.__version__, "| torchvision:", torchvision.__version__)
if EXPECTED_PYTHON is not None and Path(sys.executable).resolve() != Path(EXPECTED_PYTHON).resolve():
    raise RuntimeError("Pilih interpreter Python yang sesuai EXPECTED_PYTHON.")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA tidak tersedia. Pilih kernel dengan PyTorch CUDA dan GPU NVIDIA.")
DEVICE = torch.device(DEVICE_REQUEST)
print("Device:", DEVICE, "|", torch.cuda.get_device_name(DEVICE))

# Dataset
<a id="dataset"></a>
<a id="1-load-data"></a>

## [1] Lokasi dataset dan inventaris gambar

Baca file citra dari folder fisik `train`, `val`, dan `test`. Metadata sistem tidak dihitung.
Label `NORMAL=0` dan `PNEUMONIA=1`; nama file menjadi sumber perkiraan subtipe dan ID pasien.

In [ ]:
for split in SPLIT_NAMES:
    for label in CLASS_NAMES:
        folder = ROOT / split / label
        if not folder.is_dir():
            raise FileNotFoundError(f"Folder dataset tidak ditemukan: {folder}")
print("ROOT:", ROOT)


In [ ]:
rows = []
for split in SPLIT_NAMES:
    for label in CLASS_NAMES:
        for p in sorted((ROOT / split / label).rglob("*")):
            if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS:
                rows.append({"path": str(p), "orig_split": split, "label": label})

df = pd.DataFrame(rows)
df["y"] = df["label"].map(LABEL2ID)   # KENAPA: model butuh angka 0/1, bukan string

# Perkiraan kelompok sumber dari nama, mis. NORMAL-1064313-0001.jpeg atau BACTERIA-2034017-0005.jpeg.
fname = df["path"].apply(os.path.basename)
df["subtype"] = np.where(fname.str.contains("bacteria", case=False), "bacteria",
                 np.where(fname.str.contains("virus", case=False), "virus", "normal"))
df["patient"] = fname.str.extract(r"((?:NORMAL|BACTERIA|VIRUS)-\d+|person\d+|IM-\d+)", expand=False, flags=re.IGNORECASE)   # kelompok sumber perkiraan, bukan ID terverifikasi

print("Total citra:", len(df))
y = df["y"].to_numpy(dtype=np.int64)
display(df.head())


## [2] Tetapkan indeks split dan validasi jumlah data

Split mengikuti folder lokal: train 4.646, validation 586, dan test 624 gambar.
Validation telah diambil secara stratified dari train sebelumnya; test bawaan dipertahankan.
Sel ini hanya membaca pembagian yang sudah ada, tanpa memindahkan atau membagi ulang file.

In [ ]:
train_idx = df.index[df["orig_split"] == "train"].to_numpy()
val_idx = df.index[df["orig_split"] == "val"].to_numpy()
test_idx = df.index[df["orig_split"] == "test"].to_numpy()

splits = {"train": train_idx, "val": val_idx, "test": test_idx}
summary = pd.DataFrame({
    s: {"total": len(ix),
        "NORMAL": int((y[ix] == 0).sum()),
        "PNEUMONIA": int((y[ix] == 1).sum()),
        "% pneumonia": round(100 * (y[ix] == 1).mean(), 1)}
    for s, ix in splits.items()
}).T
print(summary)

assert df["path"].is_unique
assert not (set(train_idx) & set(val_idx))
assert not (set(train_idx) & set(test_idx))
assert not (set(val_idx) & set(test_idx))
assert len(train_idx) + len(val_idx) + len(test_idx) == len(df)
assert (len(train_idx), len(val_idx), len(test_idx)) == tuple(EXPECTED_SPLIT_COUNTS[s] for s in SPLIT_NAMES)
print("\nOK: split mengikuti folder lokal dan setiap path masuk tepat satu subset.")


# Exploratory Data Analysis
<a id="eda"></a>
<a id="2-eda"></a>

Eksplorasi berikut memeriksa ketimpangan kelas, variasi ukuran dan intensitas, contoh citra,
serta potensi irisan pasien. Hasilnya menjelaskan kebutuhan preprocessing dan batas interpretasi eksperimen.

## [1] Distribusi kelas dan subtipe pneumonia

In [ ]:
ct = pd.crosstab(df["orig_split"], df["label"], margins=True)
print(ct)

total = df["label"].value_counts()
print("\nProporsi kelas (seluruh data):")
print((total / total.sum()).round(3))
print("\nRasio Pneumonia : Normal =", round(total["PNEUMONIA"] / total["NORMAL"], 2), ": 1")

# Klaim paper
print("\nKlaim paper  : Normal=3907, Pneumonia=1949")
print("Data asli    : Normal=%d, Pneumonia=%d" % (total["NORMAL"], total["PNEUMONIA"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# (a) per kelas keseluruhan
sns.countplot(data=df, x="label", order=CLASS_NAMES, hue="label", palette="Set2", legend=False, ax=axes[0])
axes[0].set_title("Distribusi kelas (seluruh data)")
for c in axes[0].containers:
    axes[0].bar_label(c)

# (b) per folder asli x kelas
sns.countplot(data=df, x="orig_split", order=["train", "val", "test"], hue="label",
              hue_order=CLASS_NAMES, palette="Set2", ax=axes[1])
axes[1].set_title("Distribusi kelas per folder lokal")
for c in axes[1].containers:
    axes[1].bar_label(c)

plt.tight_layout(); plt.show()
# KENAPA countplot: otomatis menghitung frekuensi tiap kategori, tidak perlu groupby manual.
# KENAPA bar_label: menampilkan angka di atas batang agar mudah dibaca saat presentasi.


In [ ]:
print(df[df.label == "PNEUMONIA"]["subtype"].value_counts())
sns.countplot(data=df[df.label == "PNEUMONIA"], x="subtype", hue="orig_split")
plt.title("Subtipe pneumonia per folder asli"); plt.show()

## [2] Ukuran gambar dan rasio aspek

Ukur lebar, tinggi, serta rasio aspek gambar asli. Variasi rasio aspek menjadi alasan menggunakan
letterbox agar bentuk anatomi dipertahankan saat membentuk input persegi.

In [ ]:
def image_dimensions(path):
    with Image.open(path) as image:
        return image.size

sizes = [image_dimensions(p) for p in tqdm(df["path"], desc="Ukuran citra")]
df["w"] = [s[0] for s in sizes]
df["h"] = [s[1] for s in sizes]
df["aspect"] = df["w"] / df["h"]

print(df[["w", "h", "aspect"]].describe().round(2))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(data=df, x="w", hue="label", bins=40, ax=axes[0]); axes[0].set_title("Lebar (px)")
sns.histplot(data=df, x="h", hue="label", bins=40, ax=axes[1]); axes[1].set_title("Tinggi (px)")
sns.histplot(data=df, x="aspect", hue="label", bins=40, ax=axes[2]); axes[2].set_title("Aspect ratio (w/h)")
plt.tight_layout(); plt.show()
# KENAPA ada aspect ratio: resize paksa ke 224x224 akan mendistorsi citra yang bentuknya jauh dari persegi.

## [3] Contoh citra dan distribusi intensitas

Tampilkan sampel per kelas dengan seed tetap, kemudian ringkas intensitas sebelum preprocessing.

In [ ]:
def show_samples(label, n=EDA_SAMPLES_PER_CLASS):
    sub = df[df.label == label].sample(n, random_state=SEED)
    fig, axes = plt.subplots(1, n, figsize=(3.2 * n, 3.4))
    for ax, (_, r) in zip  (axes, sub.iterrows()):
        img = cv2.imread(r["path"], cv2.IMREAD_GRAYSCALE)   # KENAPA IMREAD_GRAYSCALE: X-ray memang 1 kanal; hemat memori & sesuai paper
        ax.imshow(img, cmap="gray")                          # KENAPA cmap="gray": tanpa ini matplotlib memberi warna palsu (viridis)
        ax.set_title(f"{label}\n{img.shape[1]}x{img.shape[0]}", fontsize=9)
        ax.axis("off")
    plt.tight_layout(); plt.show()

show_samples("NORMAL")
show_samples("PNEUMONIA")

In [ ]:
def intensity_stats(label, n=EDA_INTENSITY_SAMPLES):
    sub = df[df.label == label].sample(n, random_state=SEED)
    means, stds, hist = [], [], np.zeros(256)
    for p in sub["path"]:
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        means.append(img.mean()); stds.append(img.std())
        hist += np.bincount(img.ravel(), minlength=256)   # KENAPA bincount: hitung frekuensi tiap nilai 0..255 jauh lebih cepat dari plt.hist per citra
    return np.array(means), np.array(stds), hist / hist.sum()

stats = {c: intensity_stats(c) for c in CLASS_NAMES}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for c in CLASS_NAMES:
    m, s, h = stats[c]
    axes[0].plot(h, label=c)
    sns.kdeplot(m, ax=axes[1], label=c, fill=True, alpha=0.3)
    sns.kdeplot(s, ax=axes[2], label=c, fill=True, alpha=0.3)
axes[0].set_title("Histogram intensitas rata-rata"); axes[0].set_xlabel("Nilai piksel (0-255)")
axes[1].set_title("Distribusi mean intensitas per citra")
axes[2].set_title("Distribusi std intensitas per citra (proxy kontras)")
for a in axes: a.legend()
plt.tight_layout(); plt.show()

for c in CLASS_NAMES:
    print(c, "| mean=%.1f | std=%.1f" % (stats[c][0].mean(), stats[c][1].mean()))

## [4] Pemeriksaan potensi leakage pasien

ID diekstrak dari nama file dan hanya merupakan perkiraan; format ID tidak konsisten antar kelas/subset.
Nilai yang tidak dikenali dikeluarkan dari himpunan ID. Irisan yang muncul perlu ditinjau sebagai potensi
leakage, sedangkan irisan nol belum membuktikan independensi pasien tanpa metadata pasien resmi.

Dataset lokal memakai pola `NORMAL-<nomor>-<urutan>`, `BACTERIA-<nomor>-<urutan>`, dan `VIRUS-<nomor>-<urutan>`. Prefix dan nomor digabung menjadi kelompok sumber perkiraan. Tanpa metadata resmi, nomor ini belum dapat dipastikan sebagai identitas pasien.

Pada dataset saat pemeriksaan, 229 kelompok sumber perkiraan muncul pada train dan validation. Pembagian fisik yang sudah ada dipertahankan; ini adalah split per gambar, bukan split pasien yang terverifikasi. Hasil EDA ini perlu ikut dicatat saat melaporkan eksperimen.

In [ ]:
per_patient = df.groupby("patient").size()
print("Jumlah pasien unik (perkiraan):", per_patient.shape[0])
print("Rata-rata citra per pasien   :", round(per_patient.mean(), 2))
print("Maks citra per pasien        :", per_patient.max())

# Pasien yang muncul di lebih dari satu folder asli
multi = df.groupby("patient")["orig_split"].nunique()
print("Pasien yang tersebar di >1 folder asli:", (multi > 1).sum())

In [ ]:
def patient_set(indices):
    # ID yang tidak terdeteksi tidak diperlakukan sebagai satu pasien bersama.
    return set(df.iloc[indices]["patient"].dropna())

for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
    print(f"Irisan ID perkiraan {left}/{right}:", len(patient_set(splits[left]) & patient_set(splits[right])))
print("Citra tanpa ID yang dikenali:", int(df["patient"].isna().sum()))


# Preprocessing dan augmentasi
<a id="preprocessing"></a>
<a id="3-preprocessing-dan-augmentasi"></a>

Bentuk kedua kondisi dari sumber dan indeks split yang sama. Semua transformasi acak hanya diterapkan pada train.

## [1] Dua jalur preprocessing: baseline dan CLAHE

Dua jalur disiapkan untuk membandingkan baseline dengan paket CLAHE + augmentasi:

| Array | Pemrosesan | Skala / tipe |
|---|---|---|
| `X_train` | grayscale -> letterbox 224x224 -> normalisasi | 0-1 / float32 |
| `X_train_clahe_aug` | grayscale -> CLAHE -> letterbox -> kebijakan 50:25:25 -> normalisasi | 0-1 / float32 |

Baseline tidak memakai CLAHE atau augmentasi acak. Pada jalur kedua, CLAHE diterapkan pada semua gambar;
50% tidak mendapat operasi acak tambahan, 25% mendapat rotasi, dan 25% Gaussian blur.
Kedua array mempertahankan urutan sumber dan label yang sama. Perbandingan ini mengukur pengaruh gabungan CLAHE dan augmentasi.

CLAHE memakai `clipLimit=2.5` dan `tileGridSize=(8,8)`, sebelum padding letterbox.

Dasar pemilihan CLAHE:

- [Qiu et al. (2023), *IRCM-Caps: An X-ray image detection method for COVID-19*](https://doi.org/10.1111/crj.13599),
  bagian 3.2 dan Tabel 2, memakai parameter tersebut dan melaporkan akurasi 92,15% menjadi 93,04%,
  serta AUC 0,9117 menjadi 0,9217 setelah CLAHE pada klasifikasi COVID-19, pneumonia viral, dan normal.
  [PDF lokal](<Paper/Qiu et al 2023 - IRCM-Caps An X-ray image detection method for COVID-19.pdf>).
- [Buriboev et al. (2025), *Binary Classification of Pneumonia in Chest X-Ray Images Using Modified Contrast-Limited Adaptive Histogram Equalization Algorithm*](https://doi.org/10.3390/s25133976),
  Tabel 5, melaporkan akurasi 86,4% pada gambar original dan 93,5% dengan CLAHE standar.
  Notebook ini memakai CLAHE standar OpenCV; metode modified CLAHE dalam paper tidak diimplementasikan.
  [PDF lokal](<Paper/Buriboev et al 2025 - Binary Classification of Pneumonia in Chest X-Ray Images Using Modified Contrast-Limited Adaptive Histogram Equalization Algorithm.pdf>).

Hasil tersebut mendasari percobaan CLAHE, tetapi belum menjamin peningkatan pada ResNet50 pretrained.
Parameter CLAHE diterapkan konsisten pada jalur CLAHE untuk train, val, dan test; tidak dipilih berdasarkan test.
CLAHE dilakukan pada gambar sebelum padding, sehingga area hitam tambahan tidak masuk perhitungan histogram CLAHE.

Letterbox memakai faktor skala yang sama pada kedua sumbu, lalu menambahkan padding hitam terpusat.
Rasio aspek dipertahankan hingga pembulatan piksel; `INTER_AREA` dipakai saat memperkecil dan `INTER_LINEAR` saat memperbesar.
[Vieira et al. (2021)](https://doi.org/10.1016/j.patcog.2021.108081), bagian 3.2, juga memakai CLAHE dan zero-padding pada CXR,
termasuk input 224 x 224. Paper itu turut mengubah metode downsampling, sehingga tidak mengisolasi manfaat padding saja.

Array perantara disimpan sebagai uint8. Array model sudah dinormalisasi menjadi float32 dalam rentang 0-1. Array besar memakai NumPy memory-map pada folder cache agar tidak harus berada seluruhnya di RAM; batch hanya menambahkan kanal dan mengacak urutan.

In [ ]:
def apply_clahe(img):
    """CLAHE standar khusus jalur CLAHE, sebelum padding."""
    if img.ndim != 2 or img.dtype != np.uint8 or img.size == 0:
        raise ValueError("CLAHE membutuhkan citra grayscale uint8 yang tidak kosong.")
    transform = cv2.createCLAHE(
        clipLimit=CLAHE_CLIP_LIMIT, tileGridSize=CLAHE_TILE_GRID_SIZE,
    )
    return transform.apply(img)

def letterbox(img, size=IMG_SIZE, pad_value=0):
    """Skalakan citra grayscale secara isotropik dan pusatkan pada kanvas persegi."""
    if img.ndim != 2 or img.size == 0:
        raise ValueError("Letterbox membutuhkan citra grayscale dua dimensi yang tidak kosong.")
    if size <= 0:
        raise ValueError("Ukuran target harus positif.")
    h, w = img.shape
    scale = size / max(h, w)
    new_w = max(1, min(size, int(round(w * scale))))
    new_h = max(1, min(size, int(round(h * scale))))
    interpolation = cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR
    resized = cv2.resize(img, (new_w, new_h), interpolation=interpolation)
    canvas = np.full((size, size), pad_value, dtype=img.dtype)
    top = (size - new_h) // 2
    left = (size - new_w) // 2
    canvas[top:top + new_h, left:left + new_w] = resized
    return canvas

def load_and_resize(path, size=IMG_SIZE, use_clahe=False):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Gagal membaca {path}")
    if use_clahe:
        img = apply_clahe(img)
    return letterbox(img, size=size)

def normalize(images):
    """Normalisasi uint8 sekali; float yang sudah dalam 0-1 tidak dibagi lagi."""
    images = np.asarray(images)
    if images.dtype == np.uint8:
        return images.astype(np.float32) / 255.0
    if np.issubdtype(images.dtype, np.floating):
        if images.size and (not np.isfinite(images).all() or images.min() < 0 or images.max() > 1):
            raise ValueError("Array float harus sudah berada dalam rentang 0-1.")
        return images.astype(np.float32, copy=False)
    raise ValueError("Normalisasi menerima uint8 atau float dalam rentang 0-1.")

def cached_array(name, shape, dtype=np.float32):
    return np.lib.format.open_memmap(
        CACHE_DIR / f"{name}.npy", mode="w+", dtype=dtype, shape=shape,
    )

def validate_dataset_bundle(bundle):
    for condition in DATA_CONDITIONS:
        data = bundle[condition]
        for split, count in EXPECTED_SPLIT_COUNTS.items():
            x, y = data[split]
            if x.shape != (count, IMG_SIZE, IMG_SIZE) or x.dtype != np.float32:
                raise ValueError(f"{condition}/{split}: expected normalized float32 ({count},224,224)")
            if np.asarray(y).shape != (count,) or not np.isin(y, [0, 1]).all():
                raise ValueError(f"Invalid labels: {condition}/{split}")
    for split in ("train", "val", "test"):
        np.testing.assert_array_equal(bundle["baseline"][split][1], bundle["clahe_aug"][split][1])
    return bundle

def resolve_modeling_data(workspace, namespace=None):
    """Reuse notebook arrays, or memory-map cached .npy files; never preprocess again."""
    workspace = Path(workspace)
    namespace = {} if namespace is None else namespace
    required = ("X_train", "X_train_clahe_aug", "X_val", "X_test", "X_val_clahe", "X_test_clahe",
                "y_train", "y_val", "y_test")
    if all(key in namespace for key in required):
        a = namespace
    else:
        archive_path = PREPROCESSED_ARCHIVE
        if not archive_path.is_file():
            raise FileNotFoundError("Run preprocessing/export first: pneunet_prep.npz was not found.")
        with np.load(archive_path, allow_pickle=False) as archive:
            labels = {key: archive[key] for key in ("y_train", "y_val", "y_test")}
        array_names = required[:6]
        cache_roots = sorted(CACHE_ROOT.glob("run_*"),
                             key=lambda p: p.stat().st_mtime_ns, reverse=True)
        cache = next((p for p in cache_roots if (p / "array_manifest.json").is_file()
                      and all((p / f"{key}.npy").is_file() for key in array_names)), None)
        if cache is None:
            stat = archive_path.stat()
            cache = workspace / "tmp" / "modeling_array_cache" / f"{stat.st_size}_{stat.st_mtime_ns}"
            cache.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(archive_path) as archive:
                for key in array_names:
                    destination = cache / f"{key}.npy"
                    if not destination.exists():
                        partial = destination.with_suffix(".partial")
                        with archive.open(f"{key}.npy") as source, partial.open("wb") as target:
                            shutil.copyfileobj(source, target, length=1024 * 1024)
                        partial.replace(destination)
        a = {key: np.load(cache / f"{key}.npy", mmap_mode="r", allow_pickle=False) for key in array_names}
        a.update(labels)
    bundle = {
        "baseline": {"train": (a["X_train"], a["y_train"]),
                     "val": (a["X_val"], a["y_val"]), "test": (a["X_test"], a["y_test"])},
        "clahe_aug": {"train": (a["X_train_clahe_aug"], a["y_train"]),
                      "val": (a["X_val_clahe"], a["y_val"]), "test": (a["X_test_clahe"], a["y_test"])},
    }
    return validate_dataset_bundle(bundle)

class PreviewImages:
    def __init__(self, paths, use_clahe=False):
        self.paths = list(paths)
        self.use_clahe = use_clahe
        self.shape = (len(self.paths), IMG_SIZE, IMG_SIZE)
        self.dtype = np.dtype("uint8")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        return load_and_resize(self.paths[int(index)], use_clahe=self.use_clahe)


if REBUILD_PREPROCESSING:
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    CACHE_DIR = Path(tempfile.mkdtemp(prefix='run_', dir=CACHE_ROOT))
    X = cached_array('baseline_uint8', (len(df), IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    X_clahe_u8 = cached_array('clahe_uint8', X.shape, dtype=np.uint8)
    for (index, image_path) in enumerate(tqdm(df['path'], desc='Resize baseline dan jalur CLAHE')):
        raw = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
        if raw is None:
            raise ValueError(f'Gagal membaca {image_path}')
        X[index] = letterbox(raw)
        X_clahe_u8[index] = letterbox(apply_clahe(raw))
        if (index + 1) % 256 == 0:
            X.flush()
            X_clahe_u8.flush()
    X.flush()
    X_clahe_u8.flush()
    y = df['y'].to_numpy(dtype=np.int64)
    print('Perantara baseline:', X.shape, X.dtype)
    print('Perantara CLAHE:', X_clahe_u8.shape, X_clahe_u8.dtype)
    print('Cache:', CACHE_DIR)
else:
    CACHE_DIR = None
    X = PreviewImages(df["path"], use_clahe=False)
    X_clahe_u8 = PreviewImages(df["path"], use_clahe=True)
    y = df["y"].to_numpy(dtype=np.int64)
    print("Preview membaca citra sesuai kebutuhan; seluruh cache tidak dibentuk ulang.")


## [2] Visualisasi preprocessing dan array per split

Bandingkan gambar asli, letterbox baseline, dan CLAHE + letterbox. Setelah itu, normalisasi
ke float32 [0,1] dalam chunk kecil agar penggunaan RAM tetap terbatas.
Validation/test pada jalur CLAHE memakai CLAHE yang sama tanpa augmentasi acak.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for row, class_name in enumerate(CLASS_NAMES):
    index = df.index[df.label == class_name][0]
    raw = cv2.imread(df.loc[index, "path"], cv2.IMREAD_GRAYSCALE)
    versions = [raw, X[index], X_clahe_u8[index]]
    titles = [
        f"{class_name} - original",
        f"Baseline: letterbox {IMG_SIZE}x{IMG_SIZE}",
        f"CLAHE + letterbox {IMG_SIZE}x{IMG_SIZE}",
    ]
    for col, (image, title) in enumerate(zip(versions, titles)):
        axes[row, col].imshow(image, cmap="gray", vmin=0, vmax=255)
        axes[row, col].set_title(title, fontsize=10)
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()

xn = normalize(X[0])
print("Setelah normalisasi: min=%.3f max=%.3f dtype=%s" % (xn.min(), xn.max(), xn.dtype))


In [ ]:
def normalized_subset(source, indices, name, chunk_size=PREPROCESS_CHUNK_SIZE):
    result = cached_array(name, (len(indices), IMG_SIZE, IMG_SIZE))
    for start in range(0, len(indices), chunk_size):
        end = min(start + chunk_size, len(indices))
        result[start:end] = normalize(source[indices[start:end]])
        result.flush()
    return result



if REBUILD_PREPROCESSING:
    # Baseline: hanya resize dan normalisasi, tanpa CLAHE.
    X_train = normalized_subset(X, train_idx, "X_train")
    X_val = normalized_subset(X, val_idx, "X_val")
    X_test = normalized_subset(X, test_idx, "X_test")
    y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]

    # Evaluasi jalur CLAHE memakai CLAHE yang sama, tanpa augmentasi acak.
    X_val_clahe = normalized_subset(X_clahe_u8, val_idx, "X_val_clahe")
    X_test_clahe = normalized_subset(X_clahe_u8, test_idx, "X_test_clahe")
    print("X_train baseline:", X_train.shape, X_train.dtype, "| skala 0-1")
    print("Validation:", X_val.shape, "| validation CLAHE:", X_val_clahe.shape)
    print("Test:", X_test.shape, "| test CLAHE:", X_test_clahe.shape)

else:
    cached_bundle = resolve_modeling_data(WORKSPACE)
    X_train, y_train = cached_bundle["baseline"]["train"]
    X_val, y_val = cached_bundle["baseline"]["val"]
    X_test, y_test = cached_bundle["baseline"]["test"]
    X_train_clahe_aug, y_train_clahe_aug = cached_bundle["clahe_aug"]["train"]
    X_val_clahe = cached_bundle["clahe_aug"]["val"][0]
    X_test_clahe = cached_bundle["clahe_aug"]["test"][0]
    print("Array normalized dimuat dari cache tanpa preprocessing/training ulang.")

sp = pd.DataFrame([{"split": s, "label": CLASS_NAMES[v]} for s, ix in splits.items() for v in y[ix]])
sns.countplot(data=sp, x="split", hue="label", order=["train", "val", "test"], hue_order=CLASS_NAMES, palette="Set2")
plt.title("Distribusi kelas train/val/test lokal")
plt.show()


## [3] Augmentasi train dan distribusi 50:25:25

Jalur `X_train_clahe_aug` memakai **50% CLAHE saja, 25% CLAHE + rotasi, dan 25% CLAHE + Gaussian blur**.
Kategori tanpa operasi acak tambahan tetap menerima CLAHE dan letterbox, pada kelas NORMAL maupun PNEUMONIA. `X_train` adalah baseline terpisah yang tidak memakai CLAHE atau augmentasi acak.

- **Rotasi:** sudut acak -15 sampai +15 derajat memberi variasi orientasi ringan.
- **Gaussian blur:** sigma acak 0,5-1,5 memberi variasi ketajaman ringan. Kernel dihitung OpenCV dari sigma.
  [Rainio & Klén (2024), *Comparison of simple augmentation transformations for a convolutional neural network classifying medical images*](https://doi.org/10.1007/s11760-024-02998-5)
  menguji kedua rentang tersebut pada data medis, termasuk dataset pneumonia ini. Gaussian blur memberikan hasil kuat dalam eksperimen mereka.
- **Pemilihan satu operasi:** setiap gambar training memperoleh satu kategori: asli, rotasi, atau Gaussian blur.
  Kebijakan ini membatasi kekuatan transformasi. CLAHE sudah diterapkan sebagai preprocessing tetap pada ketiga kategori; rotasi dan Gaussian blur dipilih satu per gambar.
  [Van der Sluijs et al. (2023), *Exploring Image Augmentations for Siamese Representation Learning with Chest X-Rays*](https://arxiv.org/abs/2301.12636),
  Appendix C, menunjukkan bahwa tambahan operasi, termasuk motion blur, tidak mengungguli kombinasi terbaik pada setup self-supervised mereka.
- **Motion blur tidak digunakan:** [Kuo et al. (2021), *Recalibration of deep learning models for abnormality detection in smartphone-captured chest radiograph*](https://doi.org/10.1038/s41746-021-00393-9)
  memakainya untuk mensimulasikan foto CXR melalui smartphone. Konteks tersebut berbeda dari radiograf digital di dataset ini;
  paper tersebut belum menetapkan motion blur sebagai pilihan terbaik untuk klasifikasi pneumonia ini.

Dasar campuran asli dan augmentasi adalah skema 1:1 pada Rainio & Klén: gambar asli dipertahankan dan satu versi augmentasi dibuat per gambar.
[Athalye & Arnaout (2023)](https://doi.org/10.1371/journal.pone.0282532) juga menerapkan campuran online sekitar 50:50 per kelas pada ultrasound.
**Pembagian bagian augmentasi menjadi 25% rotasi dan 25% Gaussian blur merupakan rancangan eksperimen notebook ini;
rasio 50:25:25 belum dibuktikan optimum untuk ResNet50 pretrained oleh paper tersebut.**

Kuota dihitung pada setiap kelas. Generator online mengacak anggota kategori pada setiap epoch; array ablation menyimpan satu realisasi tetap dengan seed 42. Setiap gambar dipakai sekali per epoch.
Untuk jumlah train saat ini, pembulatan menghasilkan:

| Kelas | CLAHE saja | CLAHE + rotasi | CLAHE + Gaussian blur | Total |
|---|---:|---:|---:|---:|
| NORMAL | 599 | 300 | 299 | 1.198 |
| PNEUMONIA | 1.724 | 862 | 862 | 3.448 |
| Total | 2.323 | 1.162 | 1.161 | 4.646 |

Distribusi divisualisasikan dari fungsi perencanaan yang sama dengan generator training.
Runner modeling memakai array tetap, tanpa augmentasi ulang. Pada generator online opsional, rasio berlaku per epoch; komposisi setiap mini-batch dapat berbeda. Notebook menyediakan generator online dan array augmentasi tetap untuk ablation, tanpa menambah file gambar dataset.
Baseline dievaluasi dengan `X_val`/`X_test`; jalur CLAHE dievaluasi dengan `X_val_clahe`/`X_test_clahe`. Semua versi evaluasi tidak memakai augmentasi acak. Pilihan parameter dan kebijakan dievaluasi melalui validation set.




Array untuk ablation:

- `X_train`, `y_train`: letterbox + normalisasi saja.
- `X_train_clahe_aug`, `y_train_clahe_aug`: CLAHE sebelum letterbox, satu kategori augmentasi sesuai kuota, lalu normalisasi.
- Kedua array training berbentuk `(4646,224,224)`, bertipe float32 dengan rentang 0-1, serta mempertahankan urutan sumber dan label identik.
- `train_aug_modes` dan `train_ablation_manifest` mencatat kategori tiap sumber; realisasi array menggunakan seed 42.

`ablation_batches` memakai seed pengacakan yang sama untuk kedua array dan menambahkan dimensi kanal.
Normalisasi tidak dilakukan dua kali pada array yang sudah berskala 0-1. `train_batches` tetap tersedia untuk eksperimen augmentasi online
dengan masukan yang dipilih secara eksplisit; array materialisasi digunakan melalui `ablation_batches`.

Perbandingan kedua jalur mengukur **pengaruh gabungan CLAHE dan augmentasi**, karena baseline tidak memakai keduanya.
Peningkatan performa tidak dapat diatribusikan hanya pada augmentasi atau hanya pada CLAHE dari dua kondisi ini.

In [ ]:
train_rng = np.random.default_rng(SEED)


def gaussian_blur(img, sigma=1.0):
    if sigma <= 0:
        raise ValueError("Sigma Gaussian harus positif.")
    return cv2.GaussianBlur(
        img, (0, 0), sigmaX=float(sigma), sigmaY=float(sigma),
        borderType=cv2.BORDER_REPLICATE,
    )


def rotate(img, angle):
    h, w = img.shape
    matrix = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    return cv2.warpAffine(
        img, matrix, (w, h), flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT, borderValue=0,
    )


def augmentation_counts(n):
    """Kuota 50:25:25 dengan pembulatan sisa terbesar, tanpa mengubah total."""
    raw_counts = n * AUGMENTATION_PROBS
    counts = np.floor(raw_counts).astype(int)
    remaining = int(n - counts.sum())
    order = np.argsort(-(raw_counts - counts), kind="stable")
    counts[order[:remaining]] += 1
    return counts


def make_augmentation_plan(labels, rng):
    """Tetapkan satu mode per gambar dan pertahankan kuota pada setiap kelas."""
    labels = np.asarray(labels)
    if labels.ndim != 1:
        raise ValueError("Label harus berupa array satu dimensi.")
    modes = np.empty(len(labels), dtype=np.uint8)
    for class_id in np.unique(labels):
        class_indices = rng.permutation(np.flatnonzero(labels == class_id))
        counts = augmentation_counts(len(class_indices))
        modes[class_indices] = np.repeat(np.arange(3, dtype=np.uint8), counts)
    return modes


def random_augment(img, rng, mode=None):
    """Pilih tepat satu kategori; generator memakai kuota melalui argumen mode."""
    if mode is None:
        mode = int(rng.choice(3, p=AUGMENTATION_PROBS))
    if mode == 0:
        return img.copy()
    if mode == 1:
        return rotate(img, rng.uniform(-ROTATION_MAX_DEGREES, ROTATION_MAX_DEGREES))
    if mode == 2:
        return gaussian_blur(img, sigma=rng.uniform(*GAUSSIAN_SIGMA_RANGE))
    raise ValueError(f"Mode augmentasi tidak dikenal: {mode}")


if REBUILD_PREPROCESSING:
    # Baseline tetap tanpa CLAHE; jalur ini membaca perantara CLAHE yang terpisah.
    ablation_rng = np.random.default_rng(SEED)
    train_aug_modes = make_augmentation_plan(y_train, ablation_rng)
    X_train_clahe_aug = cached_array("X_train_clahe_aug", X_train.shape)
    for index, source_index in enumerate(tqdm(train_idx, desc="CLAHE + augmentasi untuk ablation")):
        transformed = random_augment(
            X_clahe_u8[source_index], ablation_rng, mode=int(train_aug_modes[index]),
        )
        X_train_clahe_aug[index] = normalize(transformed)
        if (index + 1) % 256 == 0:
            X_train_clahe_aug.flush()
    X_train_clahe_aug.flush()
    y_train_clahe_aug = y_train.copy()
    train_ablation_manifest = pd.DataFrame({
        "source_path": df.iloc[train_idx]["path"].to_numpy(),
        "label": [CLASS_NAMES[int(value)] for value in y_train],
        "augmentation": [AUGMENTATION_NAMES[int(mode)] for mode in train_aug_modes],
    })
    assert X_train.shape == X_train_clahe_aug.shape
    assert np.array_equal(y_train, y_train_clahe_aug)
    print("X_train: resize + normalisasi", X_train.shape, X_train.dtype)
    print("X_train_clahe_aug: CLAHE + resize + augmentasi + normalisasi", X_train_clahe_aug.shape, X_train_clahe_aug.dtype)
    print("Urutan sumber dan label sama; kedua array sudah dalam skala 0-1.")

else:
    with np.load(PREPROCESSED_ARCHIVE, allow_pickle=False) as archive:
        train_aug_modes = archive["train_aug_modes"]
        if "train_source_paths" in archive.files:
            np.testing.assert_array_equal(archive["train_source_paths"], df.iloc[train_idx]["path"].to_numpy())
    y_train_clahe_aug = y_train.copy()
    train_ablation_manifest = pd.DataFrame({
        "source_path": df.iloc[train_idx]["path"].to_numpy(),
        "label": [CLASS_NAMES[int(value)] for value in y_train],
        "augmentation": [AUGMENTATION_NAMES[int(mode)] for mode in train_aug_modes],
    })
    assert len(train_aug_modes) == len(y_train)
    print("Rencana augmentasi dimuat dari arsip yang sudah ada.")

# Grafik memakai kategori dari array augmentasi yang sudah dibentuk.
preview_modes = train_aug_modes
policy_rows = []
for class_id, class_name in enumerate(CLASS_NAMES):
    counts = np.bincount(preview_modes[y_train == class_id], minlength=3)
    policy_rows.append({
        "Kelas": class_name,
        **dict(zip(AUGMENTATION_LABELS, counts)),
        "Total": int(counts.sum()),
    })
augmentation_summary = pd.DataFrame(policy_rows).set_index("Kelas")
augmentation_summary.loc["TOTAL"] = augmentation_summary.sum()
print("Distribusi X_train_clahe_aug berdasarkan kategori augmentasi:")
print(augmentation_summary)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
total_counts = augmentation_summary.loc["TOTAL", list(AUGMENTATION_LABELS)].to_numpy(dtype=int)
bars = axes[0].bar(AUGMENTATION_LABELS, total_counts, color=sns.color_palette("Set2", 3))
axes[0].bar_label(
    bars,
    labels=[f"{n:,}\n({100 * n / len(y_train):.2f}%)" for n in total_counts],
    padding=4,
)
axes[0].set_title("Distribusi X_train_clahe_aug")
axes[0].set_ylabel("Jumlah gambar")
axes[0].set_ylim(0, max(total_counts) * 1.25)

augmentation_summary.loc[CLASS_NAMES, list(AUGMENTATION_LABELS)].plot.bar(
    ax=axes[1], color=sns.color_palette("Set2", 3), rot=0,
)
for container in axes[1].containers:
    axes[1].bar_label(container, fmt="%.0f", padding=3)
axes[1].set_title("Distribusi augmentasi pada setiap kelas")
axes[1].set_xlabel("Kelas")
axes[1].set_ylabel("Jumlah gambar")
axes[1].set_ylim(0, augmentation_summary.loc[CLASS_NAMES, AUGMENTATION_LABELS[0]].max() * 1.3)
axes[1].legend(title="Kategori")
plt.tight_layout()
plt.show()


## [4] Pemeriksaan visual rotasi dan Gaussian blur

Preview berikut membantu memeriksa kekuatan transformasi pada kedua kelas. Distribusi materialisasi
50:25:25 telah dihitung dari `train_aug_modes` pada sel sebelumnya.

In [ ]:
def show_class_augmentation(class_id, n=3, seed=SEED):
    name = CLASS_NAMES[class_id]
    local_rng = np.random.default_rng(seed)
    ids = np.where(y_train == class_id)[0]
    chosen = local_rng.choice(ids, size=n, replace=False)
    columns = ["Baseline: resize + normalisasi", "CLAHE + rotasi 12 derajat", "CLAHE + Gaussian blur"]
    fig, axes = plt.subplots(n, len(columns), figsize=(10, 3.2 * n), squeeze=False)
    for row, index in enumerate(chosen):
        img = X_clahe_u8[train_idx[index]]
        versions = [X_train[index], normalize(rotate(img, 12)), normalize(gaussian_blur(img, sigma=1.0))]
        for col, version in enumerate(versions):
            axes[row, col].imshow(version, cmap="gray", vmin=0, vmax=1)
            axes[row, col].axis("off")
            if row == 0:
                axes[row, col].set_title(columns[col], fontsize=10)
    fig.suptitle(f"{name}: perbandingan kategori augmentasi", fontsize=14)
    plt.tight_layout()
    plt.show()


show_class_augmentation(1)
show_class_augmentation(0)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for row, class_id in enumerate([1, 0]):
    index = np.where(y_train == class_id)[0][0]
    before = X_clahe_u8[train_idx[index]]
    after = gaussian_blur(before, sigma=1.0)
    axes[row, 0].imshow(before, cmap="gray", vmin=0, vmax=255)
    axes[row, 0].set_title(f"{CLASS_NAMES[class_id]} - CLAHE saja")
    axes[row, 0].axis("off")
    axes[row, 1].imshow(after, cmap="gray", vmin=0, vmax=255)
    axes[row, 1].set_title("Gaussian blur, sigma=1.0")
    axes[row, 1].axis("off")
    axes[row, 2].hist(before.ravel(), bins=64, alpha=0.6, label="CLAHE saja")
    axes[row, 2].hist(after.ravel(), bins=64, alpha=0.6, label="Gaussian blur")
    axes[row, 2].set_title("Histogram intensitas")
    axes[row, 2].legend()
plt.tight_layout()
plt.show()


## [5] Batch berpasangan dan arsip preprocessing

Contoh batch memperlihatkan bahwa label kedua kondisi tetap berpasangan. Ekspor arsip dilewati ketika
REBUILD_PREPROCESSING=False; data yang telah digunakan pada eksperimen dibaca kembali dari cache.

In [ ]:
def train_batches(X_u8, labels, batch_size=TRAIN_CONFIG["batch_size"], rng=None, return_modes=False):
    """Augmentasi online pada input yang diberikan, dengan kuota per kelas."""
    if batch_size <= 0:
        raise ValueError("Batch size harus positif.")
    labels = np.asarray(labels)
    if len(X_u8) != len(labels):
        raise ValueError("Jumlah gambar dan label harus sama.")
    rng = train_rng if rng is None else rng
    modes = make_augmentation_plan(labels, rng)
    indices = rng.permutation(len(X_u8))
    for start in range(0, len(indices), batch_size):
        batch_indices = indices[start:start + batch_size]
        xb = np.stack([
            random_augment(X_u8[i], rng, mode=int(modes[i]))
            for i in batch_indices
        ])
        xb = normalize(xb)[..., None]
        yb = labels[batch_indices].astype(np.float32)
        if return_modes:
            yield xb, yb, modes[batch_indices].copy()
        else:
            yield xb, yb


def ablation_batches(X_u8, labels, batch_size=TRAIN_CONFIG["batch_size"], seed=SEED):
    """Batch kedua jalur: urutan berpasangan, kanal ditambah, tanpa augmentasi ulang."""
    labels = np.asarray(labels)
    if batch_size <= 0 or len(X_u8) != len(labels):
        raise ValueError("Batch size harus positif; jumlah gambar dan label harus sama.")
    order = np.random.default_rng(seed).permutation(len(X_u8))
    for start in range(0, len(order), batch_size):
        indices = order[start:start + batch_size]
        yield normalize(X_u8[indices])[..., None], labels[indices].astype(np.float32)


xb_base, yb_base = next(ablation_batches(X_train, y_train))
xb_aug, yb_aug = next(ablation_batches(X_train_clahe_aug, y_train_clahe_aug))
assert np.array_equal(yb_base, yb_aug)
print("Batch baseline tanpa CLAHE/augmentasi:", xb_base.shape, xb_base.dtype)
print("Batch CLAHE + augmentasi:", xb_aug.shape, xb_aug.dtype)


In [ ]:
if REBUILD_PREPROCESSING:
    # Ekspor memakai nama jalur yang eksplisit; data sudah dinormalisasi menjadi float32.
    np.savez_compressed(
        PREPROCESSED_ARCHIVE,
        X_train=X_train, y_train=y_train,
        X_train_clahe_aug=X_train_clahe_aug, y_train_clahe_aug=y_train_clahe_aug,
        X_val=X_val, y_val=y_val, X_test=X_test, y_test=y_test,
        X_val_clahe=X_val_clahe, X_test_clahe=X_test_clahe,
        train_aug_modes=train_aug_modes,
        train_source_paths=train_ablation_manifest["source_path"].to_numpy(dtype=str),
        baseline_preprocessing=np.array(["grayscale", "letterbox", "normalize"]),
        clahe_preprocessing=np.array(["grayscale", "clahe", "letterbox", "augment_train_only", "normalize"]),
        normalization_range=np.array([0.0, 1.0]),
        resize_method=np.array("letterbox"),
        clahe_clip_limit=np.array(CLAHE_CLIP_LIMIT),
        clahe_tile_grid_size=np.array(CLAHE_TILE_GRID_SIZE),
        augmentation_seed=np.array(SEED), augmentation_modes=np.array(AUGMENTATION_NAMES),
        augmentation_probabilities=AUGMENTATION_PROBS,
        rotation_max_degrees=np.array(ROTATION_MAX_DEGREES),
        gaussian_sigma_range=np.array(GAUSSIAN_SIGMA_RANGE),
    )
    print("Tersimpan: pneunet_prep.npz (baseline resize-normalisasi dan jalur CLAHE-augmentasi)")

    cache_manifest = {
        "baseline": str(CACHE_DIR / "X_train.npy"),
        "clahe_augmented": str(CACHE_DIR / "X_train_clahe_aug.npy"),
        "shape": list(X_train.shape), "dtype": str(X_train.dtype),
        "normalization_range": [0, 1], "archive": str(PREPROCESSED_ARCHIVE),
        "augmentation_counts": np.bincount(train_aug_modes, minlength=3).tolist(),
    }
    (CACHE_DIR / "array_manifest.json").write_text(json.dumps(cache_manifest, indent=2), encoding="utf-8")
else:
    print("Ekspor preprocessing dilewati; arsip dan cache yang ada tetap dipakai.")


# Modeling dan protokol eksperimen
<a id="modeling"></a>

Bagian ini mendokumentasikan arsitektur dan konfigurasi yang menghasilkan checkpoint terpilih.
Definisi fungsi tidak melakukan training. Preview inisialisasi dilewati pada mode laporan;
bobot model untuk Grad-CAM dimuat langsung dari checkpoint full yang sesuai.

## [1] Model dan tujuan eksperimen

Eksperimen memakai **ResNet50 Torchvision dengan bobot `ResNet50_Weights.IMAGENET1K_V2`**,
GAP bawaan, dropout 0,3, dan Linear 2048 ke satu logit. Pretrained ImageNet menyediakan bobot awal;
seluruh layer backbone dan classifier tetap trainable untuk end-to-end fine-tuning.

Dua kondisi memakai sumber/label dan seed classifier yang sama. Smoke memeriksa jalur training dan
checkpoint pada GPU, tanpa menguji test set atau menyimpulkan performa model.
Salinan ini menjalankan eksperimen transfer learning; konfigurasi dan hasilnya dilaporkan sebagai
ResNet50 pretrained, dengan arsitektur standar yang dipertahankan.

## [2] Konfigurasi fine-tuning dan smoke test

| Konfigurasi | Full training | Smoke |
|---|---|---|
| Bobot awal backbone | ResNet50 IMAGENET1K_V2 | Sama, pretrained asli |
| Layer trainable | Seluruh backbone dan classifier | Sama |
| Input dataset | Grayscale letterbox 224x224, float32 [0,1] | Sama |
| Adapter model | Ulang ke RGB, mean/std ImageNet | Sama |
| Head | GAP bawaan, dropout 0,3, Linear 2048 ke 1 logit | Sama |
| Optimizer | Adam, LR 0,001; betas 0,9/0,999; epsilon 1e-7 | Sama |
| Loss | BCEWithLogitsLoss + L2 kernel 0,001 | Sama |
| Effective batch | 32 | 2 |
| Micro-batch GPU | 4 | 2 |
| Maksimum epoch | 50 | 1 |
| Early stopping | Minimum val_loss, patience 7 | Checkpoint simpan/muat diuji |
| Device / precision | CUDA / FP32 | Sama |
| Seed | 42 | 42 |

LR, dropout, dan L2 mengikuti konfigurasi notebook sumber; ini adalah konfigurasi eksperimen,
bukan hasil pencarian hyperparameter optimum untuk pretrained ResNet50. Adam memakai weight_decay=0
karena penalti L2 kernel ditambahkan eksplisit pada loss. Binary sigmoid diterapkan saat pelaporan,
sedangkan model mengeluarkan logit untuk BCEWithLogitsLoss.

Effective batch 32 diakumulasi dari micro-batch 4 untuk membatasi VRAM; statistik BatchNorm mengikuti
micro-batch, sehingga berbeda dari physical batch 32. Ubah micro_batch_size menjadi 32 jika perangkat
memadai dan eksperimen membutuhkan statistik batch tersebut. Full training tidak dijalankan saat smoke.

Referensi: [ResNet50 weights dan preprocessing resmi](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet50.html)
dan [tutorial transfer learning PyTorch](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html).

In [ ]:
REPRODUCTION_NOTES = {
    "model": "One standard Torchvision ResNet50 backbone initialized from IMAGENET1K_V2. Binary Linear head replaces ImageNet classifier; no added attention modules.",
    "transfer_learning": "End-to-end fine-tuning: all backbone and classifier parameters remain trainable. Only the new classifier is initialized with Xavier uniform and zero bias.",
    "preprocessing": "Keep the source letterbox and baseline/CLAHE+augmentation arrays in 0-1. Repeat grayscale to RGB and apply ImageNet channel normalization once inside the model.",
    "geometry": "The existing letterbox224 pipeline is retained; the weights' default resize232/center-crop224 is not applied again.",
    "framework": "PyTorch CUDA FP32; BCEWithLogitsLoss; explicit L2 Conv2d/Linear; Adam weight_decay=0.",
    "memory": "Effective batch 32 accumulated over micro-batches; BatchNorm updates per micro-batch instead of physical batch32.",
}

def configure_pytorch(seed=SEED, cpu_threads=CPU_THREADS):
    if not torch.cuda.is_available():
        raise RuntimeError("GPU CUDA wajib tersedia; CPU fallback tidak diaktifkan.")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.set_num_threads(cpu_threads)
    torch.cuda.set_device(DEVICE)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = DETERMINISTIC
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(DETERMINISTIC)
    return DEVICE


def reset_model_state(seed=SEED):
    gc.collect()
    torch.cuda.empty_cache()
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def kernel_l2(model, config=TRAIN_CONFIG):
    return config["l2"] * torch.stack([
        module.weight.square().sum() for module in model.modules()
        if isinstance(module, (nn.Conv2d, nn.Linear))
    ]).sum()


def parameter_counts(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    parameters = sum(p.numel() for p in model.parameters())
    running_statistics = sum(value.numel() for name, value in model.named_buffers()
                             if name.endswith(("running_mean", "running_var")))
    return {"parameters": parameters, "trainable_parameters": trainable,
            "bn_running_statistics": running_statistics,
            "keras_comparable_total": parameters + running_statistics}

## [3] Adapter grayscale dan normalisasi ImageNet

Array cache tetap grayscale dan berskala [0,1]. `GrayToRGB` mengulang kanal menjadi tiga,
lalu `ImageNetNormalize` memakai mean `(0.485, 0.456, 0.406)` dan std `(0.229, 0.224, 0.225)`.
Mean/std disimpan sebagai buffer agar ikut device model dan ikut checkpoint.

Normalisasi ini diterapkan **tepat sekali** pada train, validation, test, dan Grad-CAM. DataLoader
hanya menambah dimensi kanal. Pipeline letterbox yang sudah ada dipertahankan; transform default
bobot V2 berupa resize232/center-crop224 tidak diterapkan ulang. Ini mempertahankan geometri
preprocessing eksperimen dengan penyesuaian distribusi kanal untuk backbone pretrained.

In [ ]:
class GrayToRGB(nn.Module):
    def forward(self, inputs):
        if inputs.ndim != 4 or inputs.shape[1] != 1:
            raise ValueError("Expected NCHW grayscale input.")
        return inputs.repeat(1, 3, 1, 1)


class ImageNetNormalize(nn.Module):
    def __init__(self, mean=IMAGENET_MEAN, std=IMAGENET_STD):
        super().__init__()
        self.register_buffer("mean", torch.tensor(mean, dtype=torch.float32).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(std, dtype=torch.float32).view(1, 3, 1, 1))

    def forward(self, inputs):
        return (inputs - self.mean) / self.std

## [4] Arsitektur ResNet50 pretrained dan head biner

`Grayscale [0,1] -> RGB berulang -> ImageNet mean/std -> ResNet50 backbone -> GAP -> Dropout -> Linear(2048,1)`

Backbone dibangun melalui `torchvision.models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)`.
Layer fc 1000 kelas diganti Identity, lalu ditambahkan head biner baru. Hanya head baru yang diinisialisasi
Xavier uniform dengan bias nol; bobot backbone pretrained tidak diinisialisasi ulang.
Jumlah parameter trainable adalah **23.510.081**. Semua parameter backbone tetap menerima gradien.

Saat membuka checkpoint, `load_pretrained=False` membangun struktur tanpa mengunduh ulang;
seluruh bobot kemudian dimuat dari `best.pt`. Saat training baru, bobot pretrained wajib dimuat.

In [ ]:
def pretrained_weights(config=TRAIN_CONFIG):
    if config["initial_weights"] != "IMAGENET1K_V2":
        raise ValueError("This notebook requires ResNet50 IMAGENET1K_V2 pretrained weights.")
    return models.ResNet50_Weights.IMAGENET1K_V2


class ResNet50Pretrained(nn.Module):
    def __init__(self, config=None, load_pretrained=True):
        super().__init__()
        config = dict(TRAIN_CONFIG if config is None else config)
        weights = pretrained_weights(config)
        self.gray_to_rgb = GrayToRGB()
        self.input_normalize = ImageNetNormalize()
        # The official constructor loads ImageNet weights. Do not reinitialize the backbone.
        self.backbone = models.resnet50(weights=weights if load_pretrained else None)
        channels = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.classifier_dropout = nn.Dropout(config["dropout"])
        self.classifier = nn.Linear(channels, 1)
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)
        self.pretrained_weights_loaded = bool(load_pretrained)

    def forward(self, inputs):
        rgb = self.input_normalize(self.gray_to_rgb(inputs))
        features = self.backbone(rgb)
        return self.classifier(self.classifier_dropout(features))


def build_model(model_name="ResNet50_Pretrained", config=None, load_pretrained=True):
    if model_name != "ResNet50_Pretrained":
        raise ValueError(f"This notebook only supports ResNet50_Pretrained: {model_name}")
    return ResNet50Pretrained(config, load_pretrained=load_pretrained)


def build_optimizer(model, config=TRAIN_CONFIG):
    return torch.optim.Adam(model.parameters(), lr=config["learning_rate"],
                            betas=(config["beta_1"], config["beta_2"]), eps=config["epsilon"],
                            weight_decay=0.0)


def inspect_resnet_stages(model):
    names = ("gray_to_rgb", "input_normalize", "backbone.conv1", "backbone.layer1", "backbone.layer2",
             "backbone.layer3", "backbone.layer4", "backbone.avgpool", "classifier")
    shapes, handles = {}, []
    def record(name):
        def hook(module, args, output):
            shapes[name] = list(output.shape)
        return hook
    was_training = model.training
    try:
        for name in names:
            handles.append(model.get_submodule(name).register_forward_hook(record(name)))
        model.eval()
        with torch.inference_mode():
            logits = model(torch.zeros((1, 1, IMG_SIZE, IMG_SIZE), device=next(model.parameters()).device))
        assert shapes["backbone.layer4"] == [1, 2048, 7, 7]
        assert shapes["backbone.avgpool"] == [1, 2048, 1, 1]
        assert list(logits.shape) == [1, 1]
        return shapes
    finally:
        for handle in handles:
            handle.remove()
        model.train(was_training)

In [ ]:
if RUN_TRAINING:
    configure_pytorch()
    preview_model = build_model().to(DEVICE)
    print(preview_model)
    print(parameter_counts(preview_model))
    stage_table = pd.DataFrame([
        {"stage": name, "shape_NCHW": str(shape)}
        for name, shape in inspect_resnet_stages(preview_model).items()
    ])
    display(stage_table)
    del preview_model
    reset_model_state()
else:
    print("Preview/audit inisialisasi dilewati; laporan memakai bobot checkpoint full.")


## [5] Audit bobot pretrained

Fungsi berikut mencocokkan setiap tensor backbone sebelum training terhadap checkpoint resmi V2,
dengan toleransi nol. Audit ini memeriksa bahwa bobot pretrained benar-benar digunakan.
Classifier biner dikecualikan karena kelas target berbeda dari ImageNet.

In [ ]:
def audit_pretrained_backbone(model, config=TRAIN_CONFIG):
    if not model.pretrained_weights_loaded:
        raise AssertionError("Audit requires a newly initialized pretrained model.")
    reference = pretrained_weights(config).get_state_dict(progress=False, check_hash=True)
    checked = 0
    for name, value in model.backbone.state_dict().items():
        if name not in reference:
            raise AssertionError(f"Pretrained tensor is missing: {name}")
        torch.testing.assert_close(value.detach().cpu(), reference[name], rtol=0, atol=0)
        checked += 1
    del reference
    return checked

### [5.1] Pemeriksaan struktur dan fine-tuning

Pastikan fc lama sudah diganti, head memiliki satu keluaran, jumlah parameter sesuai, dan seluruh
backbone trainable. Audit juga memeriksa mean/std normalisasi dan kecocokan tensor pretrained.

In [ ]:
if RUN_TRAINING:
    check_model = build_model()
    assert isinstance(check_model.backbone.fc, nn.Identity)
    assert check_model.classifier.in_features == 2048 and check_model.classifier.out_features == 1
    assert not hasattr(check_model, "attention")
    assert all(parameter.requires_grad for parameter in check_model.backbone.parameters())
    assert all(parameter.requires_grad for parameter in check_model.classifier.parameters())
    assert sum(parameter.numel() for parameter in check_model.parameters()) == 23_510_081
    pretrained_tensors_checked = audit_pretrained_backbone(check_model)
    torch.testing.assert_close(check_model.input_normalize.mean.flatten(), torch.tensor(IMAGENET_MEAN))
    torch.testing.assert_close(check_model.input_normalize.std.flatten(), torch.tensor(IMAGENET_STD))
    display(pd.DataFrame([{"model": "ResNet50_Pretrained", **parameter_counts(check_model)}]))
    print("AUDIT PASS: pretrained tensors matched exactly:", pretrained_tensors_checked)
    print("All backbone and binary-head parameters are trainable.")
    del check_model
    reset_model_state()
else:
    print("Preview/audit inisialisasi dilewati; laporan memakai bobot checkpoint full.")


## [6] Data loader dan pasangan ablation

Loader memakai array hasil preprocessing yang sudah ada, tanpa mengulang preprocessing atau menggandakan data di RAM.
Jika variabel belum tersedia pada kernel, loader memakai cache `.npy` memory-map atau mengekstraknya secara streaming dari `pneunet_prep.npz`.

| Kondisi | Train | Validation | Test |
|---|---|---|---|
| baseline | X_train | X_val | X_test |
| clahe_aug | X_train_clahe_aug | X_val_clahe | X_test_clahe |

Label, urutan sumber, dan jumlah gambar sama: 4.646 / 586 / 624. Input sudah float32 0-1;
Dataset menambahkan kanal di depan (CHW); DataLoader mengacak train dengan generator seed tetap. Batch dipindahkan ke GPU saat dipakai. `num_workers=0` dipilih untuk notebook Windows. Normalisasi ImageNet hanya berada di model; tidak ada augmentasi kedua di DataLoader.
Smoke mengambil dua sampel, satu dari tiap kelas, hanya dari train dan validation.

In [ ]:
class ModelArrayDataset(Dataset):
    def __init__(self, images, labels):
        self.images = images
        self.labels = np.asarray(labels, dtype=np.float32).reshape(-1, 1)
        if len(images) != len(self.labels):
            raise ValueError("Invalid image/label counts.")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        image = np.array(self.images[index], dtype=np.float32, copy=True)
        if image.shape != (IMG_SIZE, IMG_SIZE) or not np.isfinite(image).all() or image.min() < 0 or image.max() > 1:
            raise ValueError("Expected normalized grayscale input in 0-1.")
        return torch.from_numpy(image[None]), torch.from_numpy(self.labels[index].copy())

def make_loader(images, labels, batch_size, shuffle=False, seed=SEED):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(ModelArrayDataset(images, labels), batch_size=batch_size, shuffle=shuffle,
                      num_workers=DATALOADER_WORKERS, pin_memory=PIN_MEMORY,
                      generator=generator, drop_last=False)

def balanced_smoke_subset(images, labels, count=2):
    if count != 2:
        raise ValueError("The smoke test is deliberately restricted to two samples.")
    indices = np.array([np.flatnonzero(np.asarray(labels) == value)[0] for value in (0, 1)])
    return np.asarray(images[indices], dtype=np.float32), np.asarray(labels)[indices]

In [ ]:
modeling_data = resolve_modeling_data(MODELING_WORKSPACE, globals())
print(pd.DataFrame([
    {"condition": condition, "split": split, "shape": str(images.shape),
     "dtype": str(images.dtype), "samples": len(labels)}
    for condition, data in modeling_data.items()
    for split, (images, labels) in data.items()
]).to_string(index=False))


## [7] Metrik evaluasi dan perbandingan kondisi

Full training menyimpan accuracy, precision/recall/F1 kedua kelas, macro F1, ROC-AUC, PR-AUC,
confusion matrix, prediksi, history, dan checkpoint. Threshold tetap 0,5 dan checkpoint dipilih
menggunakan minimum validation loss. Tabel delta membandingkan baseline dengan CLAHE+augmentasi
pada model ResNet50 pretrained yang sama. Tidak ada angka paper yang dianggap sebagai hasil run ini.

Smoke memeriksa fungsi evaluasi pada dua citra validation tanpa menyajikannya sebagai benchmark.

In [ ]:
def binary_report(labels, probabilities, threshold=0.5):
    labels = np.asarray(labels).reshape(-1).astype(int)
    probabilities = np.asarray(probabilities).reshape(-1)
    if len(labels) != len(probabilities) or not np.isfinite(probabilities).all():
        raise ValueError("Invalid predictions.")
    predictions = (probabilities >= threshold).astype(int)
    report = classification_report(labels, predictions, labels=[0, 1],
                                   target_names=["NORMAL", "PNEUMONIA"],
                                   output_dict=True, zero_division=0)
    matrix = confusion_matrix(labels, predictions, labels=[0, 1])
    metrics = {"accuracy": float(accuracy_score(labels, predictions)),
               "precision_pneumonia": report["PNEUMONIA"]["precision"],
               "recall_pneumonia": report["PNEUMONIA"]["recall"],
               "f1_pneumonia": report["PNEUMONIA"]["f1-score"],
               "precision_normal": report["NORMAL"]["precision"],
               "recall_normal": report["NORMAL"]["recall"],
               "f1_normal": report["NORMAL"]["f1-score"],
               "macro_f1": report["macro avg"]["f1-score"],
               "roc_auc": float(roc_auc_score(labels, probabilities)) if len(np.unique(labels)) == 2 else None,
               "pr_auc": float(average_precision_score(labels, probabilities)) if len(np.unique(labels)) == 2 else None,
               "samples": len(labels), "threshold": float(threshold)}
    return metrics, report, matrix, predictions


@torch.inference_mode()
def predict_loader(model, loader):
    model.eval()
    pieces = []
    for images, _ in loader:
        logits = model(images.to(DEVICE, non_blocking=PIN_MEMORY))
        pieces.append(torch.sigmoid(logits).cpu().numpy().reshape(-1))
    return np.concatenate(pieces)

## [8] Training loop, checkpoint, dan smoke

Mode full memakai train lengkap dan maksimum 50 epoch. Mode smoke memakai dua citra train dan dua
validation, satu epoch dan satu update per kondisi. Loss, gradien, bobot, dan prediksi diperiksa finite.
Checkpoint `best.pt` menyimpan state_dict, konfigurasi, epoch, model/kondisi, dan versi arsitektur.

Smoke juga memeriksa bahwa bobot convolution awal backbone berubah setelah satu update fine-tuning,
kemudian menguji pemuatan ulang checkpoint. Shape layer4, pooling, dan output dicatat dalam metadata.

In [ ]:
def initial_weight_fingerprint(model):
    digest = hashlib.sha256()
    for name, value in model.state_dict().items():
        digest.update(name.encode("utf-8"))
        digest.update(value.detach().cpu().numpy().tobytes())
    return digest.hexdigest()


def load_checkpoint(model, checkpoint):
    payload = torch.load(checkpoint, map_location="cpu", weights_only=True)
    if payload.get("architecture_version") != ARCH_VERSION:
        raise ValueError("Checkpoint tidak cocok dengan arsitektur ResNet50 pretrained pada notebook ini.")
    if payload.get("config", {}).get("initial_weights") != "IMAGENET1K_V2":
        raise ValueError("Checkpoint tidak memiliki konfigurasi pretrained yang sesuai.")
    model.load_state_dict(payload["state_dict"], strict=True)
    return {key: value for key, value in payload.items() if key != "state_dict"}


def train_one_epoch(model, loader, optimizer, config, max_batches=None):
    model.train()
    criterion = nn.BCEWithLogitsLoss()
    total_loss, total_samples, updates = 0.0, 0, 0
    actual, probabilities = [], []
    for batch_index, (images, labels) in enumerate(loader):
        if max_batches is not None and batch_index >= max_batches:
            break
        optimizer.zero_grad(set_to_none=True)
        logical_count = len(labels)
        micro_size = min(config["micro_batch_size"], logical_count)
        batch_loss = 0.0
        for start in range(0, logical_count, micro_size):
            x = images[start:start + micro_size].to(DEVICE, non_blocking=PIN_MEMORY)
            y = labels[start:start + micro_size].to(DEVICE, non_blocking=PIN_MEMORY)
            logits = model(x)
            loss = criterion(logits, y) + kernel_l2(model, config)
            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite training loss.")
            fraction = len(y) / logical_count
            (loss * fraction).backward()
            batch_loss += float(loss.detach()) * fraction
            actual.extend(y.detach().cpu().numpy().reshape(-1))
            probabilities.extend(torch.sigmoid(logits.detach()).cpu().numpy().reshape(-1))
        if max_batches is not None:
            for name, parameter in model.named_parameters():
                if parameter.grad is not None and not torch.isfinite(parameter.grad).all():
                    raise RuntimeError(f"Non-finite gradient: {name}")
        optimizer.step()
        updates += 1
        total_loss += batch_loss * logical_count
        total_samples += logical_count
    if not total_samples:
        raise RuntimeError("Empty training loader.")
    metrics, _, _, _ = binary_report(actual, probabilities, config["threshold"])
    return {"loss": total_loss / total_samples, "accuracy": metrics["accuracy"], "roc_auc": metrics["roc_auc"]}, updates


@torch.inference_mode()
def validate_one_epoch(model, loader, config, max_batches=None):
    model.eval()
    criterion = nn.BCEWithLogitsLoss()
    total_loss, count, actual, probabilities = 0.0, 0, [], []
    penalty = float(kernel_l2(model, config))
    for index, (images, labels) in enumerate(loader):
        if max_batches is not None and index >= max_batches:
            break
        x, y = images.to(DEVICE, non_blocking=PIN_MEMORY), labels.to(DEVICE, non_blocking=PIN_MEMORY)
        logits = model(x)
        loss = float(criterion(logits, y)) + penalty
        if not np.isfinite(loss):
            raise RuntimeError("Non-finite validation loss.")
        total_loss += loss * len(y)
        count += len(y)
        actual.extend(y.cpu().numpy().reshape(-1))
        probabilities.extend(torch.sigmoid(logits).cpu().numpy().reshape(-1))
    metrics, _, _, _ = binary_report(actual, probabilities, config["threshold"])
    return {"val_loss": total_loss / count, "val_accuracy": metrics["accuracy"], "val_roc_auc": metrics["roc_auc"]}


def run_experiment(model_name, condition, bundle, output_root, mode="smoke", config=None):
    if mode not in ("smoke", "full") or model_name not in MODEL_NAMES or condition not in DATA_CONDITIONS:
        raise ValueError("Invalid experiment configuration.")
    config = dict(TRAIN_CONFIG if config is None else config)
    pretrained_weights(config)
    if config["micro_batch_size"] < 1:
        raise ValueError("micro_batch_size must be positive.")
    if mode == "smoke" and (SMOKE_CONFIG != {"train_samples": 2, "validation_samples": 2, "batch_size": 2,
                                           "epochs": 1, "steps_per_epoch": 1, "validation_steps": 1}):
        raise ValueError("Smoke limits must remain two samples, one epoch, one update.")
    reset_model_state(config["seed"])
    directory = Path(output_root) / f"{model_name}__{condition}"
    directory.mkdir(parents=True, exist_ok=True)
    data = bundle[condition]
    if mode == "smoke":
        train_x, train_y = balanced_smoke_subset(*data["train"], SMOKE_CONFIG["train_samples"])
        val_x, val_y = balanced_smoke_subset(*data["val"], SMOKE_CONFIG["validation_samples"])
        batch_size, epochs = SMOKE_CONFIG["batch_size"], SMOKE_CONFIG["epochs"]
    else:
        train_x, train_y = data["train"]
        val_x, val_y = data["val"]
        batch_size, epochs = config["batch_size"], config["epochs"]
    train_loader = make_loader(train_x, train_y, batch_size, shuffle=True, seed=config["seed"])
    inference_batch = min(batch_size, config["micro_batch_size"])
    val_loader = make_loader(val_x, val_y, inference_batch)
    model = build_model(model_name, config, load_pretrained=True).to(DEVICE)
    assert model.pretrained_weights_loaded
    initial_conv1 = model.backbone.conv1.weight.detach().cpu().clone()
    optimizer = build_optimizer(model, config)
    if next(model.parameters()).device.type != "cuda":
        raise RuntimeError("Model must run on CUDA.")
    torch.cuda.reset_peak_memory_stats(DEVICE)
    metadata = {"model": model_name, "condition": condition, "mode": mode,
                "framework": "pytorch", "architecture_version": ARCH_VERSION, "torch": str(torch.__version__), "torchvision": str(torchvision.__version__),
                "python": sys.executable, "device": str(DEVICE), "gpu": torch.cuda.get_device_name(DEVICE),
                "cuda_build": torch.version.cuda, "precision": "float32", **parameter_counts(model),
                "config": config, "effective_batch_size": batch_size,
                "micro_batch_size": min(batch_size, config["micro_batch_size"]), "effective_epochs": epochs,
                "training_samples": len(train_y), "validation_samples": len(val_y),
                "initial_weight_sha256": initial_weight_fingerprint(model), "test_evaluated": False,
                "purpose": "error_detection_only" if mode == "smoke" else "performance_evaluation",
                "reproduction_notes": REPRODUCTION_NOTES}
    metadata["stage_shapes_NCHW"] = inspect_resnet_stages(model)
    metadata["pretrained_weights"] = config["initial_weights"]
    metadata["pretrained_weights_loaded"] = model.pretrained_weights_loaded
    metadata["finetune_backbone"] = True
    metadata["input_normalization"] = {"mean": list(IMAGENET_MEAN), "std": list(IMAGENET_STD)}
    (directory / "architecture.txt").write_text(str(model) + chr(10) + json.dumps(parameter_counts(model)), encoding="utf-8")
    (directory / "architecture.json").write_text(json.dumps({"model": model_name, "config": config,
            "parameters": parameter_counts(model)}, indent=2), encoding="utf-8")
    checkpoint = directory / "best.pt"
    rows, best_loss, best_epoch, stale, updates = [], float("inf"), None, 0, 0
    start = time.perf_counter()
    try:
        for epoch in range(epochs):
            train_stats, epoch_updates = train_one_epoch(model, train_loader, optimizer, config,
                                                          max_batches=1 if mode == "smoke" else None)
            validation = validate_one_epoch(model, val_loader, config, max_batches=1 if mode == "smoke" else None)
            updates += epoch_updates
            row = {"epoch": epoch, **train_stats, **validation}
            rows.append(row)
            pd.DataFrame(rows).to_csv(directory / f"{mode}_history.csv", index=False)
            if validation["val_loss"] < best_loss:
                best_loss, best_epoch, stale = validation["val_loss"], epoch + 1, 0
                torch.save({"state_dict": {key: value.detach().cpu().clone() for key, value in model.state_dict().items()},
                            "config": config, "epoch": best_epoch, "model": model_name, "condition": condition,
                             "architecture_version": ARCH_VERSION}, checkpoint)
            else:
                stale += 1
            if mode == "full":
                print(f"  epoch {epoch + 1}/{epochs}: loss={row['loss']:.5f}, val_loss={row['val_loss']:.5f}", flush=True)
            if stale >= config["early_stopping_patience"]:
                break
        torch.cuda.synchronize(DEVICE)
        metadata.update({"seconds": time.perf_counter() - start, "epochs_completed": len(rows),
                         "optimizer_updates": updates, "best_epoch": best_epoch,
                         "gpu_peak_allocated_mb": torch.cuda.max_memory_allocated(DEVICE) / 1024**2})
        if not checkpoint.is_file():
            raise RuntimeError("Best checkpoint was not saved.")
        load_checkpoint(model, checkpoint)
        probabilities = predict_loader(model, val_loader)
        if not np.isfinite(probabilities).all() or np.any((probabilities < 0) | (probabilities > 1)):
            raise RuntimeError("Invalid sigmoid predictions.")
        for name, weight in model.state_dict().items():
            if not torch.isfinite(weight).all():
                raise RuntimeError(f"Non-finite model state: {name}")
        if mode == "smoke":
            assert updates == len(rows) == 1 and len(train_y) == len(val_y) == 2
            binary_report(val_y, probabilities, config["threshold"])
            with torch.no_grad():
                model.classifier.weight.zero_()
            load_checkpoint(model, checkpoint)
            np.testing.assert_allclose(predict_loader(model, val_loader), probabilities, rtol=1e-5, atol=1e-6)
            backbone_updated = not torch.equal(model.backbone.conv1.weight.detach().cpu(), initial_conv1)
            assert backbone_updated, "Backbone was not updated during the fine-tuning smoke test."
            metadata.update({"finite_training_loss": True, "finite_validation_loss": True,
                             "backbone_updated": backbone_updated,
                             "checkpoint_reload_passed": True, "status": "PASS"})
        else:
            for split in ("train", "val", "test"):
                images, labels = data[split]
                probability = predict_loader(model, make_loader(images, labels, inference_batch))
                metrics, report, matrix, predictions = binary_report(labels, probability, config["threshold"])
                metadata[f"{split}_metrics"] = metrics
                (directory / f"{split}_classification_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
                pd.DataFrame(matrix, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(directory / f"{split}_confusion_matrix.csv")
                pd.DataFrame({"y_true": labels, "probability": probability, "prediction": predictions}).to_csv(directory / f"{split}_predictions.csv", index=False)
            metadata.update({"test_evaluated": True, "status": "COMPLETE"})
        (directory / "result.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
        return metadata
    finally:
        del model, optimizer
        reset_model_state(config["seed"])

In [ ]:
def summarize_runs(results, output_root, mode):
    rows = []
    for result in results:
        row = {key: result[key] for key in ("model", "condition", "mode", "status", "parameters", "trainable_parameters", "optimizer_updates", "seconds", "device", "gpu", "gpu_peak_allocated_mb")}
        if mode == "full":
            row.update(result["test_metrics"])
        else:
            row.update({"test_evaluated": result["test_evaluated"],
                        "checkpoint_reload_passed": result["checkpoint_reload_passed"]})
        rows.append(row)
    table = pd.DataFrame(rows)
    table.to_csv(Path(output_root) / f"{mode}_summary.csv", index=False)
    return table


def full_comparison_artifacts(results, bundle, output_root):
    """Full mode only: curves and condition comparison for one ResNet50 model."""
    from sklearn.metrics import roc_curve, precision_recall_curve
    output_root = Path(output_root)
    for condition in DATA_CONDITIONS:
        available = [result for result in results if result["condition"] == condition]
        if not available:
            continue
        fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
        labels = bundle[condition]["test"][1]
        for result in available:
            name = result["model"]
            directory = output_root / f"{name}__{condition}"
            predictions = pd.read_csv(directory / "test_predictions.csv")
            np.testing.assert_array_equal(predictions["y_true"].to_numpy(), labels)
            fpr, tpr, _ = roc_curve(labels, predictions["probability"])
            precision, recall, _ = precision_recall_curve(labels, predictions["probability"])
            axes[0].plot(fpr, tpr, label=name)
            axes[1].plot(recall, precision, label=name)
            history = pd.read_csv(directory / "full_history.csv")
            axes[2].plot(history["epoch"] + 1, history["val_loss"], label=name)
            fig_history, history_axes = plt.subplots(1, 2, figsize=(11, 4))
            for key, axis in (("accuracy", history_axes[0]), ("loss", history_axes[1])):
                axis.plot(history["epoch"] + 1, history[key], label="train")
                axis.plot(history["epoch"] + 1, history[f"val_{key}"], label="validation")
                axis.set(xlabel="Epoch", ylabel=key, title=f"{name} / {condition}")
                axis.legend()
            fig_history.tight_layout()
            fig_history.savefig(directory / "training_curves.png", dpi=150)
            plt.close(fig_history)
            matrix = confusion_matrix(labels, predictions["prediction"], labels=[0, 1])
            fig_cm, axis = plt.subplots(figsize=(4, 4))
            axis.imshow(matrix, cmap="Blues")
            for row in range(2):
                for col in range(2):
                    axis.text(col, row, str(matrix[row, col]), ha="center", va="center")
            axis.set(xticks=[0, 1], yticks=[0, 1], xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                     xlabel="Predicted", ylabel="True", title=f"{name} / {condition}")
            fig_cm.tight_layout()
            fig_cm.savefig(directory / "test_confusion_matrix.png", dpi=150)
            plt.close(fig_cm)
        for axis, title in zip(axes, ("Test ROC", "Test precision-recall", "Validation loss")):
            axis.set_title(title)
            axis.legend(fontsize=8)
        axes[0].set(xlabel="False positive rate", ylabel="True positive rate")
        axes[1].set(xlabel="Recall", ylabel="Precision")
        axes[2].set(xlabel="Epoch", ylabel="Validation loss including L2")
        fig.tight_layout()
        fig.savefig(output_root / f"full_comparison_{condition}.png", dpi=150)
        plt.close(fig)
    summary = summarize_runs(results, output_root, "full")
    paired = summary.pivot(index="model", columns="condition", values=["accuracy", "macro_f1"])
    if all(condition in paired.columns.get_level_values(1) for condition in DATA_CONDITIONS):
        deltas = pd.DataFrame({
            "accuracy_baseline": paired[("accuracy", "baseline")],
            "accuracy_clahe_aug": paired[("accuracy", "clahe_aug")],
            "accuracy_delta": paired[("accuracy", "clahe_aug")] - paired[("accuracy", "baseline")],
            "macro_f1_delta": paired[("macro_f1", "clahe_aug")] - paired[("macro_f1", "baseline")],
        })
        deltas.to_csv(output_root / "full_ablation_deltas.csv")


def run_modeling_suite(workspace, mode="smoke", model_names=MODEL_NAMES,
                       conditions=DATA_CONDITIONS, namespace=None, output_root=None):
    if mode not in ("smoke", "full"):
        raise ValueError("mode must be 'smoke' or 'full'")
    workspace = Path(workspace)
    configure_pytorch(TRAIN_CONFIG["seed"])
    bundle = resolve_modeling_data(workspace, namespace)
    output_root = Path(output_root) if output_root else MODELING_RESULTS_ROOT / f"pytorch_{mode}_{ARCH_VERSION}_{datetime.now():%Y%m%d_%H%M%S}"
    output_root.mkdir(parents=True, exist_ok=True)
    results = []
    for name in model_names:
        for condition in conditions:
            print(f"{mode.upper()}: {name} / {condition}", flush=True)
            result = run_experiment(name, condition, bundle, output_root, mode)
            results.append(result)
            summarize_runs(results, output_root, mode)
            print(f"  {result['status']}; updates={result['optimizer_updates']}; test_evaluated={result['test_evaluated']}", flush=True)
    table = summarize_runs(results, output_root, mode)
    if mode == "full":
        full_comparison_artifacts(results, bundle, output_root)
    print("Results:", output_root)
    return table, output_root

## [9] Eksekusi laporan dari artefak tersimpan

`RUN_TRAINING=False` membuat sel eksekusi membaca result.json, full_history.csv, dan test_predictions.csv.
Metrik diverifikasi terhadap prediksi yang disimpan. Grafik berikut menampilkan train loss dan validation loss
untuk setiap model, pada skala linear serta log agar fase awal dan akhir training sama-sama terbaca.
Tidak ada pemanggilan run_modeling_suite pada mode laporan.

# Hasil evaluasi tersimpan
<a id="hasil-evaluasi"></a>

Sel berikut membaca metadata, prediksi, dan history dari run full yang dipilih. Metrik dicocokkan dengan prediksi tersimpan; optimizer tidak dipanggil.

In [ ]:
def saved_experiment_records(run_dir=None):
    run_dir = SAVED_RUN_DIR if run_dir is None else Path(run_dir)
    if not Path(run_dir).is_dir():
        raise FileNotFoundError("Artefak full belum tersedia. Isi SAVED_RUN_DIR; baca README bagian laporan tersimpan.")
    records = []
    for model_name in MODEL_NAMES:
        for condition in DATA_CONDITIONS:
            directory = run_dir / f"{model_name}__{condition}"
            info = json.loads((directory / "result.json").read_text(encoding="utf-8"))
            if info.get("mode") != "full" or info.get("status") != "COMPLETE" or not info.get("test_evaluated"):
                raise ValueError(f"Hasil full belum lengkap: {directory}")
            if info["model"] != model_name or info["condition"] != condition:
                raise ValueError(f"Model/kondisi tidak cocok: {directory}")
            expected_version = globals().get("ARCH_VERSION")
            if expected_version is not None and info.get("architecture_version") != expected_version:
                raise ValueError(f"Versi arsitektur checkpoint berbeda: {directory}")
            history = pd.read_csv(directory / "full_history.csv")
            required = {"epoch", "loss", "val_loss"}
            if not required.issubset(history.columns) or history.empty:
                raise ValueError(f"History tidak lengkap: {directory}")
            if not np.isfinite(history[["loss", "val_loss"]].to_numpy()).all():
                raise ValueError(f"Loss non-finite: {directory}")
            prediction = pd.read_csv(directory / "test_predictions.csv")
            metrics, _, _, _ = binary_report(prediction.y_true, prediction.probability, info["config"]["threshold"])
            for key in ("accuracy", "macro_f1", "recall_normal", "recall_pneumonia"):
                np.testing.assert_allclose(metrics[key], info["test_metrics"][key], atol=1e-10)
            records.append({"directory": directory, "info": info, "history": history, "prediction": prediction})
    return records


def saved_results_table(records):
    rows = []
    for record in records:
        info = record["info"]
        rows.append({"Model": info["model"], "Kondisi": info["condition"],
                     "Accuracy (%)": 100 * info["test_metrics"]["accuracy"],
                     "Macro-F1": info["test_metrics"]["macro_f1"],
                     "ROC-AUC": info["test_metrics"]["roc_auc"],
                     "Recall NORMAL": info["test_metrics"]["recall_normal"],
                     "Recall PNEUMONIA": info["test_metrics"]["recall_pneumonia"],
                     "Epoch terbaik": info.get("best_epoch"), "Epoch selesai": info.get("epochs_completed"),
                     "Parameter": info["trainable_parameters"]})
    table = pd.DataFrame(rows)
    table.to_csv(REPORT_OUTPUT_DIR / "saved_results_summary.csv", index=False)
    return table


def display_saved_loss_curves(records):
    files = []
    for model_name in MODEL_NAMES:
        selected = [record for record in records if record["info"]["model"] == model_name]
        fig, axes = plt.subplots(2, len(selected), figsize=(6.7 * len(selected), 7.8), squeeze=False)
        for column, record in enumerate(selected):
            history, info = record["history"], record["info"]
            epochs = history["epoch"].to_numpy() + 1
            for row in range(2):
                axis = axes[row, column]
                axis.plot(epochs, history["loss"], label="Train loss", color="#2563eb", linewidth=1.8)
                axis.plot(epochs, history["val_loss"], label="Validation loss", color="#dc2626", linewidth=1.8)
                if info.get("best_epoch") is not None:
                    axis.axvline(info["best_epoch"], color="#64748b", linestyle="--", alpha=0.7,
                                 label=f"Checkpoint epoch {info['best_epoch']}")
                axis.set(xlabel="Epoch", ylabel="BCE + L2 loss", title=f"{info['condition']} | {'linear' if row == 0 else 'detail log'}")
                if row == 1:
                    if (history[["loss", "val_loss"]].to_numpy() > 0).all():
                        axis.set_yscale("log")
                    else:
                        axis.set_yscale("symlog", linthresh=0.01)
                axis.legend(fontsize=8)
                axis.grid(alpha=0.25)
        fig.suptitle(f"{model_name}: loss dari history training tersimpan", fontsize=14)
        fig.tight_layout(rect=(0, 0, 1, 0.96))
        destination = REPORT_OUTPUT_DIR / f"{model_name}_train_validation_loss.png"
        fig.savefig(destination, dpi=150, bbox_inches="tight")
        plt.show()
        plt.close(fig)
        files.append(str(destination))
    return files


def display_saved_benchmark():
    file = Path(modeling_output_dir) / "test_inference_accuracy_f1_latency_batch1.csv"
    if file.is_file():
        display(pd.read_csv(file).round(4))
    else:
        print("Benchmark latency tidak tersedia pada run ini; tidak dihitung ulang.")

In [ ]:
if RUN_TRAINING:
    modeling_summary, modeling_output_dir = run_modeling_suite(
        MODELING_WORKSPACE, mode=MODELING_RUN_MODE,
        model_names=SELECTED_MODELS, conditions=SELECTED_CONDITIONS,
        namespace=globals(),
    )
    print(modeling_summary.to_string(index=False))
else:
    modeling_output_dir = SAVED_RUN_DIR
    saved_records = saved_experiment_records(modeling_output_dir)
    modeling_summary = saved_results_table(saved_records)
    print("Hasil dimuat dari:", modeling_output_dir)
    print("RUN_TRAINING=False: tidak ada pemanggilan training.")


In [ ]:
if globals().get("RUN_TRAINING", False) and globals().get("MODELING_RUN_MODE") == "smoke":
    print("Laporan full dilewati pada smoke; tidak mengevaluasi test set.")
else:
    if RUN_TRAINING:
        saved_records = saved_experiment_records(modeling_output_dir)
        modeling_summary = saved_results_table(saved_records)
    display(modeling_summary.round(4))
    loss_plot_files = display_saved_loss_curves(saved_records)


## Confusion matrix dan classification report seluruh eksperimen
<a id="classification-reports"></a>

Bagian ini menampilkan **setiap model x kondisi baseline/CLAHE+augmentasi x split train/validation/test**.
Sumbernya hanya `{split}_predictions.csv` dan `result.json` dari run full yang dipilih;
checkpoint dan citra tidak perlu dimuat. Seluruh angka dicocokkan kembali dengan metrik tersimpan.

**Cara menjalankan tanpa training:** cukup jalankan tiga cell kode di bagian ini secara berurutan.
Cell pertama mengimpor pustaka dan memiliki fallback path run untuk notebook ini, sehingga dapat dijalankan
pada kernel baru tanpa menjalankan cell modeling atau preprocessing di atasnya.
Jika konfigurasi paling atas sudah dijalankan, `SAVED_RUN_DIR`, `MODEL_NAMES`, `REPORT_OUTPUT_DIR`,
`CLASSIFICATION_REPORT_SPLITS`, dan `CLASSIFICATION_REPORT_EXPORT` mengikuti nilai konfigurasi tersebut.
Untuk hanya menampilkan test, ubah `CLASSIFICATION_REPORT_SPLITS = ("test",)` pada konfigurasi paling atas.

Urutan kelas tetap **0 = NORMAL, 1 = PNEUMONIA**. Baris confusion matrix adalah kelas aktual;
kolom adalah prediksi. Karena PNEUMONIA adalah kelas positif, posisi matriks adalah `[[TN, FP], [FN, TP]]`.
Angka anotasi adalah jumlah citra dan persentase per kelas aktual; warna juga memakai proporsi per baris
agar split dengan ukuran berbeda tetap mudah dibandingkan.

Classification report memuat precision, recall, F1-score, support, macro average, dan weighted average;
accuracy ditampilkan terpisah. `zero_division=0` mempertahankan kelas yang tidak diprediksi (misalnya
NORMAL pada VGG16) tanpa menyembunyikannya. **Metrik test menjadi acuan performa generalisasi**;
train/validation adalah evaluasi pada split tersebut, bukan hasil out-of-fold. Pada kondisi CLAHE+aug,
prediksi train mengikuti data augmentasi tetap yang digunakan saat evaluasi run tersimpan.


In [ ]:
if globals().get("RUN_TRAINING", False) and globals().get("MODELING_RUN_MODE") == "smoke":
    print("Laporan full dilewati pada smoke; tidak mengevaluasi test set.")
else:
    # Baca prediksi tersimpan saja; cell ini dapat dijalankan pada kernel baru.
    from pathlib import Path
    import json
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
    from IPython.display import Markdown, display

    classification_run_dir = Path(globals().get("modeling_output_dir", globals().get("SAVED_RUN_DIR", (Path.cwd() / "outputs" / "modeling" / "saved_full_v3"))))
    classification_output_dir = Path(globals().get("REPORT_OUTPUT_DIR", (Path.cwd() / "outputs" / "notebook_reports" / "resnet50_pretrained_v3"))) / "classification_reports"
    classification_models = tuple(globals().get("MODEL_NAMES", ('ResNet50_Pretrained',)))
    classification_conditions = ("baseline", "clahe_aug")
    classification_splits = tuple(globals().get("CLASSIFICATION_REPORT_SPLITS", ("train", "val", "test")))
    classification_export = bool(globals().get("CLASSIFICATION_REPORT_EXPORT", True))
    classification_class_names = ("NORMAL", "PNEUMONIA")
    classification_condition_names = {"baseline": "Baseline", "clahe_aug": "CLAHE + augmentasi"}
    classification_split_names = {"train": "Train", "val": "Validation", "test": "Test"}

    if not classification_splits or len(set(classification_splits)) != len(classification_splits):
        raise ValueError("CLASSIFICATION_REPORT_SPLITS harus berisi split yang unik dan tidak kosong.")
    if not set(classification_splits).issubset(classification_split_names):
        raise ValueError("Split yang didukung: train, val, test.")


    def load_saved_classification_records(run_dir, model_names, splits):
        records, summary_rows, report_rows, matrix_rows = [], [], [], []
        labels_by_split = {}
        for model_name in model_names:
            for condition in classification_conditions:
                directory = Path(run_dir) / f"{model_name}__{condition}"
                result_path = directory / "result.json"
                info = json.loads(result_path.read_text(encoding="utf-8"))
                if info.get("mode") != "full" or info.get("status") != "COMPLETE":
                    raise ValueError(f"Eksperimen full belum lengkap: {directory}")
                if info.get("model") != model_name or info.get("condition") != condition:
                    raise ValueError(f"Identitas eksperimen tidak cocok: {directory}")
                threshold = float(info["config"]["threshold"])
                for split in splits:
                    prediction_path = directory / f"{split}_predictions.csv"
                    frame = pd.read_csv(prediction_path)
                    required = {"y_true", "prediction", "probability"}
                    if frame.empty or not required.issubset(frame.columns):
                        raise ValueError(f"CSV prediksi kosong/tidak lengkap: {prediction_path}")
                    numeric = frame[list(sorted(required))].apply(pd.to_numeric, errors="raise")
                    if not np.isfinite(numeric.to_numpy()).all():
                        raise ValueError(f"Nilai non-finite pada {prediction_path}")
                    if not frame["y_true"].isin([0, 1]).all() or not frame["prediction"].isin([0, 1]).all():
                        raise ValueError(f"Label harus 0=NORMAL atau 1=PNEUMONIA: {prediction_path}")
                    y_true = frame["y_true"].to_numpy(dtype=int)
                    y_pred = frame["prediction"].to_numpy(dtype=int)
                    probability = frame["probability"].to_numpy(dtype=float)
                    if ((probability < 0) | (probability > 1)).any():
                        raise ValueError(f"Probabilitas di luar [0,1]: {prediction_path}")
                    # Kolom prediction adalah hasil keputusan tersimpan. Toleransi hanya
                    # mengakomodasi pembulatan probabilitas CSV sangat dekat threshold.
                    inconsistent = (y_pred != (probability >= threshold).astype(int)) & (np.abs(probability - threshold) > 1e-7)
                    if inconsistent.any():
                        raise ValueError(f"Prediksi tidak sesuai threshold tersimpan: {prediction_path}")
                    expected = info[f"{split}_metrics"]
                    if len(frame) != int(expected["samples"]):
                        raise ValueError(f"Jumlah sampel berbeda dari result.json: {prediction_path}")
                    if split in labels_by_split:
                        np.testing.assert_array_equal(y_true, labels_by_split[split], err_msg=f"Label split berbeda: {prediction_path}")
                    else:
                        labels_by_split[split] = y_true.copy()
                    report = classification_report(y_true, y_pred, labels=[0, 1],
                        target_names=list(classification_class_names), output_dict=True, zero_division=0)
                    matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])
                    accuracy = float(accuracy_score(y_true, y_pred))
                    observed = {"accuracy": accuracy, "macro_f1": report["macro avg"]["f1-score"],
                        "precision_normal": report["NORMAL"]["precision"], "recall_normal": report["NORMAL"]["recall"],
                        "f1_normal": report["NORMAL"]["f1-score"],
                        "precision_pneumonia": report["PNEUMONIA"]["precision"],
                        "recall_pneumonia": report["PNEUMONIA"]["recall"], "f1_pneumonia": report["PNEUMONIA"]["f1-score"]}
                    for metric, value in observed.items():
                        np.testing.assert_allclose(value, expected[metric], rtol=0, atol=1e-10,
                            err_msg=f"{model_name}/{condition}/{split}: {metric} berbeda dari artefak")
                    identity = {"model": model_name, "condition": condition, "split": split}
                    tn, fp, fn, tp = (int(value) for value in matrix.ravel())
                    summary_rows.append({**identity, "samples": len(frame), "threshold": threshold,
                        **observed, "TN": tn, "FP": fp, "FN": fn, "TP": tp})
                    record = {**identity, "samples": len(frame), "threshold": threshold, "accuracy": accuracy,
                        "report": report, "matrix": matrix, "prediction_path": prediction_path}
                    records.append(record)
                    for name in (*classification_class_names, "macro avg", "weighted avg"):
                        values = report[name]
                        report_rows.append({**identity, "class": name, "precision": values["precision"],
                            "recall": values["recall"], "f1-score": values["f1-score"],
                            "support": int(values["support"]), "accuracy": accuracy, "threshold": threshold})
                    for actual in range(2):
                        support = int(matrix[actual].sum())
                        for predicted in range(2):
                            matrix_rows.append({**identity, "actual": classification_class_names[actual],
                                "predicted": classification_class_names[predicted], "count": int(matrix[actual, predicted]),
                                "fraction_of_actual_class": float(matrix[actual, predicted] / support) if support else 0.0})
        return records, pd.DataFrame(summary_rows), pd.DataFrame(report_rows), pd.DataFrame(matrix_rows)


    (classification_records, classification_summary,
     classification_reports, classification_matrices) = load_saved_classification_records(
        classification_run_dir, classification_models, classification_splits)
    print("Sumber prediksi:", classification_run_dir)
    print(f"{len(classification_records)} kombinasi model/kondisi/split telah diverifikasi terhadap result.json.")
    print("CSV saja: tidak memuat checkpoint, menjalankan inferensi, atau memanggil training.")


### Seluruh confusion matrix

Kolom kiri menampilkan baseline; kolom kanan CLAHE+augmentasi. Setiap baris menunjukkan satu split.

In [ ]:
if globals().get("RUN_TRAINING", False) and globals().get("MODELING_RUN_MODE") == "smoke":
    print("Laporan full dilewati pada smoke; tidak mengevaluasi test set.")
else:
    # Seluruh confusion matrix: kolom = kondisi, baris = split, satu figure per model.
    def plot_saved_confusion_matrices(records, model_names, splits, output_dir=None):
        files = []
        lookup = {(r["model"], r["condition"], r["split"]): r for r in records}
        for model_name in model_names:
            fig, axes = plt.subplots(len(splits), 2, figsize=(11.8, 4.0 * len(splits)),
                squeeze=False, layout="constrained")
            for row, split in enumerate(splits):
                for column, condition in enumerate(classification_conditions):
                    record = lookup[(model_name, condition, split)]
                    matrix = record["matrix"]
                    support = matrix.sum(axis=1, keepdims=True)
                    normalized = np.divide(matrix, support, out=np.zeros_like(matrix, dtype=float), where=support != 0)
                    axis = axes[row, column]
                    image = axis.imshow(normalized, cmap="Blues", vmin=0, vmax=1)
                    for actual in range(2):
                        for predicted in range(2):
                            axis.text(predicted, actual, f"{matrix[actual, predicted]:,}\n({normalized[actual, predicted]:.1%})",
                                ha="center", va="center", fontsize=12,
                                color="white" if normalized[actual, predicted] > 0.5 else "#0f172a")
                    axis.set_xticks([0, 1], classification_class_names)
                    axis.set_yticks([0, 1], classification_class_names)
                    axis.set(xlabel="Prediksi", ylabel="Aktual",
                        title=f"{classification_condition_names[condition]} | {classification_split_names[split]} (n={record['samples']:,})\n"
                              f"Accuracy={record['accuracy']:.2%} | F1 macro={record['report']['macro avg']['f1-score']:.4f}")
            fig.suptitle(f"{model_name}: confusion matrix dari prediksi tersimpan", fontsize=15)
            fig.colorbar(image, ax=axes.ravel().tolist(), shrink=0.65, label="Proporsi per kelas aktual")
            if output_dir is not None:
                output_dir = Path(output_dir)
                output_dir.mkdir(parents=True, exist_ok=True)
                destination = output_dir / f"{model_name}_confusion_matrices.png"
                fig.savefig(destination, dpi=150, bbox_inches="tight")
                files.append(str(destination))
            display(fig)
            plt.close(fig)
        return files


    classification_plot_files = plot_saved_confusion_matrices(
        classification_records, classification_models, classification_splits,
        classification_output_dir if classification_export else None)


### Seluruh classification report

Report dipisahkan berdasarkan model, kondisi, dan split. Rekap TN/FP/FN/TP serta metrik ditampilkan setelah seluruh report.

In [ ]:
if globals().get("RUN_TRAINING", False) and globals().get("MODELING_RUN_MODE") == "smoke":
    print("Laporan full dilewati pada smoke; tidak mengevaluasi test set.")
else:
    # Classification report lengkap: setiap model, kondisi dataset, dan split.
    for classification_model in classification_models:
        display(Markdown(f"### {classification_model}"))
        for classification_condition in classification_conditions:
            display(Markdown(f"#### {classification_condition_names[classification_condition]}"))
            for classification_split in classification_splits:
                selected = classification_reports.loc[
                    (classification_reports["model"] == classification_model)
                    & (classification_reports["condition"] == classification_condition)
                    & (classification_reports["split"] == classification_split)]
                details = classification_summary.loc[
                    (classification_summary["model"] == classification_model)
                    & (classification_summary["condition"] == classification_condition)
                    & (classification_summary["split"] == classification_split)].iloc[0]
                display(Markdown(f"**{classification_split_names[classification_split]}** | "
                    f"n={int(details['samples']):,} | accuracy={details['accuracy']:.2%} | threshold={details['threshold']:g}"))
                report_table = selected.set_index("class")[["precision", "recall", "f1-score", "support"]]
                display(report_table.style.format({"precision": "{:.4f}", "recall": "{:.4f}",
                    "f1-score": "{:.4f}", "support": "{:,.0f}"}))

    # Tabel rekap membantu membandingkan semua eksperimen tanpa menelusuri setiap report.
    display(Markdown("### Rekap confusion matrix dan metrik seluruh eksperimen"))
    display(classification_summary[["model", "condition", "split", "samples", "TN", "FP", "FN", "TP",
        "accuracy", "macro_f1", "recall_normal", "recall_pneumonia"]].style.format({
            "accuracy": "{:.2%}", "macro_f1": "{:.4f}", "recall_normal": "{:.2%}", "recall_pneumonia": "{:.2%}"}))

    if classification_export:
        classification_output_dir.mkdir(parents=True, exist_ok=True)
        classification_summary.to_csv(classification_output_dir / "classification_metrics_summary.csv", index=False)
        classification_reports.to_csv(classification_output_dir / "classification_reports_all.csv", index=False)
        classification_matrices.to_csv(classification_output_dir / "confusion_matrices_all.csv", index=False)
        print("Salinan tabel dan figure:", classification_output_dir)


# Analisis kesalahan dan Grad-CAM
<a id="gradcam"></a>

## [1] NORMAL dan PNEUMONIA pada setiap model

Setiap model ditampilkan pada kedua kondisi preprocessing. Dua citra test dipilih secara deterministik:
file pertama berlabel NORMAL dan file pertama berlabel PNEUMONIA pada urutan inventaris yang sama.
Citra yang sama dipakai antar-model sehingga galeri tidak hanya memilih contoh prediksi yang benar.

Setiap baris menampilkan citra input, overlay target NORMAL, dan overlay target PNEUMONIA.
Probabilitas hasil checkpoint dicocokkan dengan prediksi test tersimpan. Heatmap nol diberi label;
heatmap menunjukkan kontribusi relatif terhadap skor kelas, bukan segmentasi atau probabilitas piksel.

In [ ]:
def error_analysis_cases(model, directory, bundle, condition, mode, source_frame):
    split = "test" if mode == "full" else "val"
    images, labels = bundle[condition][split]
    source_rows = source_frame.loc[source_frame["orig_split"] == split].reset_index(drop=True)
    np.testing.assert_array_equal(source_rows["y"].to_numpy(), labels)
    if mode == "full":
        predictions_path = Path(directory) / "test_predictions.csv"
        if not predictions_path.is_file():
            raise FileNotFoundError("Prediksi full/test belum tersedia; jalankan full training sendiri terlebih dahulu.")
        table = pd.read_csv(predictions_path)
        np.testing.assert_array_equal(table["y_true"].to_numpy(), labels)
        indices = np.arange(len(labels))
        table.insert(0, "source_index", indices)
    elif mode == "smoke":
        indices = np.array([np.flatnonzero(np.asarray(labels) == value)[0] for value in (0, 1)])
        sample = np.asarray(images[indices], dtype=np.float32)
        sample_labels = np.asarray(labels)[indices]
        probability = predict_loader(model, make_loader(sample, sample_labels, len(indices)))
        table = pd.DataFrame({"source_index": indices, "y_true": np.asarray(labels)[indices],
                              "probability": probability,
                              "prediction": (probability >= TRAIN_CONFIG["threshold"]).astype(int)})
    else:
        raise ValueError("Unknown mode.")
    if not np.isfinite(table["probability"]).all():
        raise ValueError("Probabilitas tidak finite.")
    table["source_path"] = source_rows.iloc[indices]["path"].to_numpy()
    table["split"] = split
    table["true_label"] = table["y_true"].map(dict(enumerate(CLASS_NAMES)))
    table["predicted_label"] = table["prediction"].map(dict(enumerate(CLASS_NAMES)))
    table["error_type"] = np.select(
        [(table.y_true == 0) & (table.prediction == 1),
         (table.y_true == 1) & (table.prediction == 0),
         (table.y_true == 0) & (table.prediction == 0)], ["FP", "FN", "TN"], default="TP")
    table["prediction_confidence"] = np.where(table.prediction == 1, table.probability, 1 - table.probability)
    table["preview_only"] = mode == "smoke"
    return table, images


def select_error_examples(table, maximum=ERROR_ANALYSIS_MAX_CASES):
    # Ambil FP/FN bergantian agar galeri tidak didominasi satu jenis kesalahan.
    queues = [table.loc[table.error_type == kind].sort_values("prediction_confidence", ascending=False)
              for kind in ("FP", "FN")]
    chosen = []
    for position in range(max(len(queue) for queue in queues)):
        for queue in queues:
            if position < len(queue) and len(chosen) < maximum:
                chosen.append(queue.index[position])
    if chosen:
        return table.loc[chosen].copy()
    return table.sort_values("prediction_confidence", ascending=False).head(maximum).copy()

## [2] Cara membaca Grad-CAM

Gradien dihitung terhadap feature map convolutional yang tercantum pada konfigurasi.
Target PNEUMONIA memakai logit pneumonia; target NORMAL memakai negatif logit tersebut.
Bobot kanal berasal dari rata-rata spasial gradien, kemudian ReLU dan normalisasi membentuk heatmap.

Checkpoint dimuat dalam evaluation mode. Hook dan status gradien dipulihkan setelah pemakaian;
optimizer dan training loop tidak dipanggil. Overlay berada pada citra letterbox yang sudah diproses
sesuai kondisi model sehingga koordinatnya tetap sejajar.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap


def make_gradcam_components(model, model_name):
    target_name = GRADCAM_TARGET_LAYERS[model_name]
    model.get_submodule(target_name)
    return {"model": model, "target_layer": target_name}


def gradcam_heatmap(image, components, class_id):
    if class_id not in (0, 1):
        raise ValueError("Target must be NORMAL=0 or PNEUMONIA=1.")
    image = np.asarray(image, dtype=np.float32)
    if image.shape != (IMG_SIZE, IMG_SIZE) or not np.isfinite(image).all() or image.min() < 0 or image.max() > 1:
        raise ValueError("Expected one normalized grayscale image.")
    model = components["model"]
    target = model.get_submodule(components["target_layer"])
    batch = torch.from_numpy(image.copy()[None, None]).to(DEVICE)
    was_training = model.training
    parameter_flags = [parameter.requires_grad for parameter in model.parameters()]
    captured = {}
    def capture(module, inputs, output):
        activation = output.detach().requires_grad_(True)
        captured["activation"] = activation
        return activation
    handle = None
    try:
        model.eval()
        for parameter in model.parameters():
            parameter.requires_grad_(False)
        handle = target.register_forward_hook(capture)
        with torch.enable_grad():
            logits = model(batch)
            score = (logits if class_id == 1 else -logits).sum()
            features = captured["activation"]
            gradients = torch.autograd.grad(score, features, retain_graph=False)[0]
            if not torch.isfinite(gradients).all():
                raise RuntimeError("Non-finite Grad-CAM gradients.")
            weights = gradients.mean(dim=(2, 3), keepdim=True)
            heatmap = F.relu((features.detach() * weights).sum(dim=1, keepdim=True))
            maximum = heatmap.amax(dim=(2, 3), keepdim=True)
            heatmap = heatmap / maximum.clamp_min(torch.finfo(heatmap.dtype).tiny)
            heatmap = F.interpolate(heatmap, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
            heatmap = heatmap[0, 0].detach().cpu().numpy()
            probability = float(torch.sigmoid(logits.detach())[0, 0])
        handle.remove()
        handle = None
        with torch.inference_mode():
            original_probability = float(torch.sigmoid(model(batch))[0, 0])
        np.testing.assert_allclose(probability, original_probability, rtol=1e-4, atol=1e-6)
        if heatmap.shape != image.shape or not np.isfinite(heatmap).all():
            raise RuntimeError("Invalid Grad-CAM heatmap.")
        return heatmap, probability
    finally:
        if handle is not None:
            handle.remove()
        for parameter, flag in zip(model.parameters(), parameter_flags):
            parameter.requires_grad_(flag)
        model.train(was_training)



def display_gradcam_examples(table, images, components, destination, mode):
    selected = select_error_examples(table)
    count = len(selected)

    if not count:
        raise ValueError("Tidak ada citra untuk Grad-CAM.")

    paper_cmap = LinearSegmentedColormap.from_list(
        "paper_cam",
        [
            "#182079",
            "#2f3eae",
            "#456cca",
            "#27b5b6",
            "#f2d76a",
            "#ed5144"
        ]
    )

    fig, axes = plt.subplots(
        count, 3,
        figsize=(16, 4.5 * count),
        squeeze=False
    )

    output_rows = []

    for row, (_, case) in enumerate(selected.iterrows()):
        image = np.asarray(
            images[int(case.source_index)],
            dtype=np.float32
        )

        heatmap, probability = gradcam_heatmap(
            image, components, int(case.prediction)
        )

        np.testing.assert_allclose(
            probability, case.probability,
            rtol=1e-4, atol=1e-6
        )

        zero_map = bool(np.max(heatmap) == 0)

        label = (
            f"{case.error_type} | "
            f"true={case.true_label}, "
            f"pred={case.predicted_label}"
        )

        # Kolom 1: Original Image
        axes[row, 0].imshow(
            image, cmap="gray", vmin=0, vmax=1
        )
        axes[row, 0].set_title(
            label + f"\np(pneumonia)={probability:.4f}",
            fontsize=10
        )

        # Kolom 2: Grad-CAM
        axes[row, 1].imshow(
            heatmap, cmap=paper_cmap,
            vmin=0, vmax=1
        )
        axes[row, 1].set_title(
            "Grad-CAM relatif" +
            (" | heatmap nol" if zero_map else "")
        )

        # Kolom 3: Overlay seperti paper
        axes[row, 2].imshow(
            image, cmap="gray",
            vmin=0, vmax=1
        )

        if not zero_map:
            overlay = axes[row, 2].imshow(
                heatmap,
                cmap=paper_cmap,
                vmin=0,
                vmax=1,
                alpha=0.42,
                interpolation="bilinear"
            )

            fig.colorbar(
                overlay,
                ax=axes[row, 2],
                fraction=0.046,
                pad=0.04
            )

        axes[row, 2].set_title(
            f"Overlay target: {case.predicted_label}"
        )

        for axis in axes[row]:
            axis.axis("off")

        output_rows.append({
            **case.to_dict(),
            "gradcam_target_class": int(case.prediction),
            "gradcam_target_layer": components["target_layer"],
            "zero_heatmap": zero_map
        })

    plt.tight_layout()

    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)

    fig.savefig(
        destination / f"{mode}_gradcam_paper.png",
        dpi=250,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    return pd.DataFrame(output_rows)

## [3] Galeri dari checkpoint full tersimpan

Galeri berikut mencakup seluruh model dan kondisi pada run terpilih. Tabel akhir mencatat label asli,
kelas target, probabilitas tersimpan, probabilitas checkpoint, serta penanda heatmap nol.
Sel ini melakukan inferensi dan perhitungan gradien untuk visualisasi saja, tanpa update bobot.

In [ ]:
def run_error_analysis(output_root, bundle, source_frame, mode=MODELING_RUN_MODE,
                       model_name=ERROR_ANALYSIS_MODEL, condition=ERROR_ANALYSIS_CONDITION):
    directory = Path(output_root) / f"{model_name}__{condition}"
    result_path = directory / "result.json"
    if not result_path.is_file():
        print(f"Error analysis dilewati: hasil {model_name}/{condition} tidak ada di run ini.")
        print("Sesuaikan ERROR_ANALYSIS_MODEL dan ERROR_ANALYSIS_CONDITION pada sel konfigurasi.")
        return None
    metadata = json.loads(result_path.read_text(encoding="utf-8"))
    if metadata.get("framework") != "pytorch":
        raise ValueError("Checkpoint harus berasal dari run PyTorch notebook ini.")
    if metadata.get("architecture_version") != ARCH_VERSION:
        raise ValueError("Artefak tidak kompatibel dengan ResNet50 pretrained pada notebook ini.")
    if metadata["mode"] != mode:
        raise ValueError("Mode checkpoint berbeda dari mode analisis.")
    if mode == "full" and not metadata["test_evaluated"]:
        raise ValueError("Checkpoint belum memiliki evaluasi full/test.")
    config = metadata["config"]
    if config["image_size"] != IMG_SIZE or config["threshold"] != TRAIN_CONFIG["threshold"]:
        raise ValueError("Ukuran input/threshold harus sesuai dengan konfigurasi checkpoint.")
    reset_model_state(config["seed"])
    model = build_model(model_name, config, load_pretrained=False).to(DEVICE)
    try:
        load_checkpoint(model, directory / "best.pt")
        table, images = error_analysis_cases(model, directory, bundle, condition, mode, source_frame)
        destination = directory / "error_analysis"
        destination.mkdir(parents=True, exist_ok=True)
        table.to_csv(destination / f"{mode}_prediction_cases.csv", index=False)
        print(f"{mode.upper()} / {model_name} / {condition} / split={table.split.iloc[0]}")
        print("Jenis hasil:", table.error_type.value_counts().to_dict())
        if mode == "smoke":
            assert len(table) == 2 and set(table.split) == {"val"}
            print("PREVIEW TEKNIS: tidak ada inferensi test atau training tambahan.")
        display(table[["source_path", "true_label", "predicted_label", "probability", "error_type"]])
        components = make_gradcam_components(model, model_name)
        selected = display_gradcam_examples(table, images, components, destination, mode)
        del components
        print("Artefak error analysis:", destination)
        return {"cases": table, "gradcam_cases": selected, "directory": destination}
    finally:
        del model
        reset_model_state(config["seed"])

In [ ]:
def saved_gradcam_gallery(records, bundle, source_frame):
    import inspect
    source_rows = source_frame.loc[source_frame.orig_split == "test"].reset_index(drop=True)
    labels = np.asarray(bundle["baseline"]["test"][1])
    np.testing.assert_array_equal(source_rows.y.to_numpy(), labels)
    sample_indices = [int(np.flatnonzero(labels == class_id)[0]) for class_id in (0, 1)]
    all_cases, plot_files = [], []
    configure_pytorch()
    for record in records:
        info, directory = record["info"], record["directory"]
        model_name, condition = info["model"], info["condition"]
        config = info["config"]
        images, current_labels = bundle[condition]["test"]
        np.testing.assert_array_equal(labels, current_labels)
        np.testing.assert_array_equal(record["prediction"].y_true.to_numpy(), labels)
        reset_model_state(config["seed"])
        kwargs = {"load_pretrained": False} if "load_pretrained" in inspect.signature(build_model).parameters else {}
        model = build_model(model_name, config, **kwargs).to(DEVICE)
        try:
            load_checkpoint(model, directory / "best.pt")
            model.eval()
            components = make_gradcam_components(model, model_name)
            fig, axes = plt.subplots(2, 3, figsize=(13, 8), squeeze=False)
            saved_maps = []
            for row, index in enumerate(sample_indices):
                image = np.asarray(images[index], dtype=np.float32)
                true_id = int(labels[index])
                prediction = record["prediction"].iloc[index]
                axes[row, 0].imshow(image, cmap="gray", vmin=0, vmax=1)
                axes[row, 0].set_title(f"Label: {CLASS_NAMES[true_id]} | pred: {CLASS_NAMES[int(prediction.prediction)]}\np(PNEUMONIA) tersimpan={prediction.probability:.4f}", fontsize=10)
                maps = []
                for target in (0, 1):
                    heatmap, probability = gradcam_heatmap(image, components, target)
                    np.testing.assert_allclose(probability, float(prediction.probability), rtol=1e-3, atol=2e-4,
                                               err_msg=f"Input/checkpoint tidak cocok: {model_name}/{condition}/{index}")
                    if not np.isfinite(heatmap).all() or heatmap.shape != (IMG_SIZE, IMG_SIZE):
                        raise ValueError("Grad-CAM tidak valid.")
                    zero_map = bool(heatmap.max() == 0)
                    axis = axes[row, target + 1]
                    axis.imshow(image, cmap="gray", vmin=0, vmax=1)
                    overlay = axis.imshow(heatmap, cmap="jet", vmin=0, vmax=1, alpha=0.65 * heatmap)
                    axis.set_title(f"Target: {CLASS_NAMES[target]}" + (" | CAM nol" if zero_map else ""))
                    fig.colorbar(overlay, ax=axis, fraction=0.046, pad=0.03)
                    all_cases.append({"model": model_name, "condition": condition, "source_index": index,
                                      "source_path": source_rows.iloc[index]["path"], "true_label": CLASS_NAMES[true_id],
                                      "target_label": CLASS_NAMES[target], "stored_probability": float(prediction.probability),
                                      "checkpoint_probability": probability, "absolute_difference": abs(probability - float(prediction.probability)),
                                      "zero_heatmap": zero_map, "checkpoint": str(directory / "best.pt"),
                                      "target_layer": components["target_layer"]})
                    maps.append(heatmap)
                saved_maps.append(maps)
                for axis in axes[row]:
                    axis.axis("off")
            fig.suptitle(f"{model_name} | {condition}: NORMAL vs PNEUMONIA\nCheckpoint full tersimpan; tidak ada training ulang", fontsize=13)
            fig.tight_layout(rect=(0, 0, 1, 0.94))
            destination = REPORT_OUTPUT_DIR / f"{model_name}__{condition}_gradcam_normal_pneumonia.png"
            fig.savefig(destination, dpi=150, bbox_inches="tight")
            np.savez_compressed(REPORT_OUTPUT_DIR / f"{model_name}__{condition}_heatmaps.npz",
                                heatmaps=np.asarray(saved_maps), sample_indices=np.asarray(sample_indices))
            plt.show()
            plt.close(fig)
            plot_files.append(str(destination))
            del components
            print(f"Grad-CAM OK: {model_name} / {condition}")
        finally:
            del model
            reset_model_state(config["seed"])
    cases = pd.DataFrame(all_cases)
    cases.to_csv(REPORT_OUTPUT_DIR / "gradcam_normal_pneumonia_cases.csv", index=False)
    return cases, plot_files

In [ ]:
if globals().get("RUN_TRAINING", False) and globals().get("MODELING_RUN_MODE") == "smoke":
    print("Laporan full dilewati pada smoke; tidak mengevaluasi test set.")
else:
    gradcam_cases, gradcam_plot_files = saved_gradcam_gallery(saved_records, modeling_data, df)
    display(gradcam_cases[["model", "condition", "true_label", "target_label", "zero_heatmap", "absolute_difference"]].round(6))

# Efisiensi inferensi tersimpan
<a id="benchmark"></a>

Tabel berikut dibaca dari CSV historis run yang sama. Benchmark tidak dijalankan ulang. Ringkasan per model, jumlah parameter, dan trade-off performa tersedia pada [kesimpulan](#kesimpulan).

Pengukuran historis menggunakan **NVIDIA GeForce RTX 3060 Laptop GPU, PyTorch CUDA FP32, input 224 x 224, batch size 1, 20 langkah warm-up, dan 624 citra test**. Latency adalah waktu per gambar, bukan waktu training.

- **GPU mean**: rata-rata waktu forward model yang diukur dengan CUDA Events; tidak mencakup transfer tensor CPU ke GPU.
- **GPU P95**: persentil ke-95 latency forward, untuk melihat variasi waktu respons; bukan interval kepercayaan.
- **Tensor-E2E mean**: mulai dari tensor CPU siap pakai, transfer ke GPU, forward, sigmoid, sampai hasil kembali ke CPU. Pembacaan/dekode gambar, letterbox, CLAHE, dan pembuatan augmentasi berada di luar timer ini.
- **Throughput**: jumlah citra per detik dari pengukuran Tensor-E2E secara berurutan pada batch 1; bukan throughput maksimum pada batch besar.

Karena preprocessing tidak masuk timer, selisih latency baseline dan CLAHE+aug tidak dapat ditafsirkan sebagai biaya komputasi CLAHE. Perbedaan kecil juga dapat dipengaruhi variasi kondisi GPU selama pengukuran. Angka ini belum mewakili latency aplikasi dari file X-ray mentah sampai prediksi.


In [ ]:
display_saved_benchmark()


# Catatan interpretasi dan batas reproduksi

Notebook ini mempertahankan inventaris data, EDA, baseline, CLAHE+augmentasi, dan split fisik dari
notebook sumber. Pemrosesan input disesuaikan dengan pretrained ResNet50 melalui normalisasi ImageNet.
Perbandingan kedua kondisi mengukur pengaruh gabungan CLAHE dan augmentasi pada fine-tuning ResNet50.
Metadata bobot awal, versi model, device, dan seed disimpan agar hasil dapat ditelusuri.

Smoke hanya mendeteksi error pada jalur yang diuji. Metrik performa disimpulkan dari full training
dan evaluasi yang dijalankan pengguna secara terpisah. Pemeriksaan kelompok sumber pada EDA tetap
perlu dicatat karena split saat ini berdasarkan gambar, bukan ID pasien yang terverifikasi.

# Kesimpulan
<a id="kesimpulan"></a>

Laporan ini menilai keseimbangan performa klasifikasi, ukuran model, dan kecepatan inferensi dari hasil yang sudah tersimpan. Metrik test memakai 624 citra dan checkpoint yang sebelumnya dipilih melalui validation loss. Satuan **pp** berarti poin persentase.

### Performa klasifikasi

| Model | Accuracy baseline | Accuracy CLAHE+aug | Selisih | Macro-F1 baseline | Macro-F1 CLAHE+aug |
|---|---:|---:|---:|---:|---:|
| ResNet50_Pretrained | 80.13% | 83.65% | +3.53 pp | 0.7555 | 0.8093 |

### Latency inferensi setiap model

| Model | Kondisi | Parameter (juta) | GPU mean (ms) | GPU P95 (ms) | Tensor-E2E mean (ms) | Throughput (citra/detik) |
|---|---|---:|---:|---:|---:|---:|
| ResNet50_Pretrained | Baseline | 23.510 | 11.42 | 16.90 | 11.70 | 85.45 |
| ResNet50_Pretrained | CLAHE+aug | 23.510 | 11.54 | 17.18 | 11.82 | 84.60 |

Parameter merupakan jumlah parameter model PyTorch, bukan FLOPs atau total buffer BatchNorm. Angka dibulatkan untuk tampilan; nilai lengkap tersedia pada [Ringkasan hasil historis](docs/experiment_results.csv) dan `result.json` per eksperimen.

Pengukuran historis menggunakan **NVIDIA GeForce RTX 3060 Laptop GPU, PyTorch CUDA FP32, input 224 x 224, batch size 1, 20 langkah warm-up, dan 624 citra test**. Latency adalah waktu per gambar, bukan waktu training.

- **GPU mean**: rata-rata waktu forward model yang diukur dengan CUDA Events; tidak mencakup transfer tensor CPU ke GPU.
- **GPU P95**: persentil ke-95 latency forward, untuk melihat variasi waktu respons; bukan interval kepercayaan.
- **Tensor-E2E mean**: mulai dari tensor CPU siap pakai, transfer ke GPU, forward, sigmoid, sampai hasil kembali ke CPU. Pembacaan/dekode gambar, letterbox, CLAHE, dan pembuatan augmentasi berada di luar timer ini.
- **Throughput**: jumlah citra per detik dari pengukuran Tensor-E2E secara berurutan pada batch 1; bukan throughput maksimum pada batch besar.

Karena preprocessing tidak masuk timer, selisih latency baseline dan CLAHE+aug tidak dapat ditafsirkan sebagai biaya komputasi CLAHE. Perbedaan kecil juga dapat dipengaruhi variasi kondisi GPU selama pengukuran. Angka ini belum mewakili latency aplikasi dari file X-ray mentah sampai prediksi.

### Interpretasi performa, ukuran, dan kecepatan

**ResNet50 pretrained CLAHE+aug memberikan accuracy 83,65% dan macro-F1 0,8093**, dibanding baseline 80,13% dan 0,7555. Kenaikan accuracy sebesar **3,53 poin persentase** disertai peningkatan recall NORMAL dari **49,15% ke 61,11%**, sementara recall PNEUMONIA berubah dari **98,72% ke 97,18%**. Perubahan ini menunjukkan keseimbangan prediksi antar-kelas yang perlu diperiksa bersama confusion matrix.

Kedua kondisi memakai arsitektur yang sama dengan **23.510.081 parameter**. Latency forward GPU tercatat **11,42 ms/gambar** pada baseline dan **11,54 ms/gambar** pada CLAHE+aug; Tensor-E2E masing-masing **11,70** dan **11,82 ms/gambar**. Selisih GPU sekitar **0,12 ms** tidak membuktikan bahwa CLAHE menambah biaya sebesar itu, karena CLAHE tidak masuk timer dan waktu pengukuran dapat bervariasi.

Eksperimen ini memberikan acuan model pretrained yang sudah di-fine-tune, untuk menilai apakah PneuNet dapat mempertahankan performa dengan lebih sedikit parameter dan latency lebih rendah. Notebook ini hanya memuat satu arsitektur; kesimpulan bahwa PneuNet menyamai atau mengungguli pretrained ResNet50 perlu membandingkan notebook pendamping dengan memperhatikan protokol training dan pengukuran. Angka latency dari run terpisah bersifat deskriptif, bukan benchmark bersama yang mengendalikan kondisi GPU. Tidak ada eksperimen ViT atau validasi inferensi pada perangkat edge dalam run ini.

### Kaitan dengan tujuan paper

Bagian **1.3 (halaman 4)** pada [paper PneuNet](https://doi.org/10.3389/fmed.2025.1713587) menempatkan kontribusi utama pada performa diagnostik yang kompetitif dengan biaya komputasi lebih rendah. Karena itu, keberhasilan eksperimen perlu dinilai bersama melalui **accuracy, macro-F1/recall per kelas, jumlah parameter, dan latency**, bukan hanya accuracy tertinggi.

**Tabel 5 (halaman 11)** melaporkan PneuNet 1,8 juta parameter, accuracy 91%, dan inference 20 ms; ResNet-50 + Attention 25,6 juta parameter, accuracy 93%, dan 48 ms. Paper mengukur waktu pada **Raspberry Pi 4, RAM 4 GB, CPU-only, TensorFlow Lite**. Latency GPU notebook ini tidak dapat dibandingkan langsung dengan 20 ms pada paper untuk menyatakan percepatan terhadap paper. FLOPs dan latency perangkat edge implementasi ini belum diukur.

### Batas kesimpulan dan arah pengembangan

Perbandingan baseline dan CLAHE+aug mengubah CLAHE dan augmentasi secara bersamaan, sehingga kontribusi keduanya belum terpisah. Hasil berasal dari satu run/split tersimpan, tanpa pengulangan seed atau uji signifikansi; perbedaan kecil tidak membuktikan keunggulan yang stabil. Grad-CAM membantu meninjau kontribusi spasial, tetapi bukan ukuran akurasi atau segmentasi.

Pengembangan berikutnya perlu mempertahankan atau meningkatkan macro-F1 dan recall NORMAL sambil menekan parameter dan latency, lalu mengevaluasi seluruh kandidat dalam protokol benchmark bersama. Untuk menguji tujuan deployment paper, diperlukan pengukuran pada perangkat target dan waktu pipeline lengkap termasuk preprocessing; pengukuran tersebut belum dilakukan dalam laporan ini.

**Tidak ada training, benchmark inferensi, atau eksekusi cell ulang pada pembaruan abstrak dan kesimpulan ini.** Seluruh angka dibaca dari artefak historis; grafik loss dan Grad-CAM yang sudah tersedia tetap dipertahankan.


# Referensi
<a id="references"></a>


- [ResNet50 Torchvision](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet50.html): bobot V2 dan normalisasi ImageNet.
- [Transfer learning PyTorch](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html): fine-tuning model pretrained.
- [Catatan augmentasi dan resize](<Paper/Catatan referensi augmentasi dan resize.md>): referensi preprocessing yang dipertahankan.
- [Grad-CAM](https://arxiv.org/abs/1610.02391): visualisasi kontribusi spasial.
- [Paper PneuNet](https://doi.org/10.3389/fmed.2025.1713587): Bagian 1.3 halaman 4 dan Tabel 5 halaman 11 menjadi rujukan tujuan CNN ringan serta evaluasi performa dan biaya inferensi.

- [Paper PneuNet](https://doi.org/10.3389/fmed.2025.1713587): sumber yang dibaca untuk memperjelas abstrak dan kesimpulan.
- [Ringkasan hasil historis](docs/experiment_results.csv): sumber GPU mean/P95, Tensor-E2E, dan throughput batch 1; jumlah parameter dan metrik diverifikasi terhadap `result.json`.
